# Exposure Micro-batch Tools

Utilities to:
1. Select a step range and collect micro-batches by rank from `exposures_rank*.jsonl`.
2. Decode any selected micro-batch using the default GPT-2 tokenizer.


In [ ]:
from __future__ import annotations

import json
import re
from functools import lru_cache
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional

import numpy as np
from transformers import AutoTokenizer


In [ ]:
# Adjust to a local completed BOS run before executing.
RUN_DIR = Path("runs/research/bos_aligned_proto/<run_name>")
EXPOSURES_DIR = RUN_DIR / "exposures"

print("RUN_DIR:", RUN_DIR)
print("EXPOSURES_DIR:", EXPOSURES_DIR)
print("Exists:", EXPOSURES_DIR.exists())


## Mental Model: Exposure Logs -> Records -> Micro-batches

Think of the exposure data as a 3-layer structure:

- Layer 1: one file per rank (`exposures_rank0000.jsonl`, ...)
- Layer 2: one JSON record per logged optimizer step (for this run, every 100 steps)
- Layer 3: `micro_batches` list inside each record

Each micro-batch entry tells you where training data came from:

- which shard file (`shard_path`, `shard_idx`)
- which block (`block_idx`)
- exact token-offset span in that shard (`start`, `end`)

So the flow is:

1. select ranks and steps
2. pull matching micro-batch descriptors
3. map descriptor offsets back to shard tokens
4. decode token rows into text


In [ ]:
RANK_FILE_RE = re.compile(r"exposures_rank(\d{4})\.jsonl$")


def discover_exposure_files(exposures_dir: Path) -> Dict[int, Path]:
    """
    Discover exposure log files and map them to rank ids.

    Expected filename format:
      exposures_rank0000.jsonl
      exposures_rank0001.jsonl
      ...

    Parameters
    ----------
    exposures_dir:
        Directory containing `exposures_rank*.jsonl` files.

    Returns
    -------
    Dict[int, Path]
        Dictionary keyed by integer rank id. Example:
        `{0: Path(.../exposures_rank0000.jsonl), 1: Path(...), ...}`

    Notes
    -----
    - Keys are parsed from the 4-digit rank suffix in the filename.
    - If no files are found, we raise `FileNotFoundError` immediately,
      because every downstream function depends on these logs.
    """
    # Normalize to Path in case caller passed a string.
    exposures_dir = Path(exposures_dir)

    out: Dict[int, Path] = {}

    # Scan files in deterministic order so outputs are reproducible.
    for p in sorted(exposures_dir.glob("exposures_rank*.jsonl")):
        m = RANK_FILE_RE.search(p.name)
        if not m:
            # Ignore files that match the glob but not strict rank naming.
            continue
        rank = int(m.group(1))
        out[rank] = p

    if not out:
        raise FileNotFoundError(f"No exposures_rank*.jsonl files found in: {exposures_dir}")

    return out


def iter_exposure_records(path: Path) -> Iterable[Dict[str, Any]]:
    """
    Stream JSONL exposure records from a single rank file.

    Why streaming:
    - Exposure files are typically small-to-medium, but streaming keeps
      memory usage stable and works well even if files get large.

    Yields
    ------
    Dict[str, Any]
        One parsed JSON object per non-empty line.
    """
    with Path(path).open("r") as f:
        for line in f:
            line = line.strip()
            if not line:
                # Skip blank lines defensively.
                continue
            yield json.loads(line)


def get_microbatches_by_rank(
    exposures_dir: Path,
    step_start: int,
    step_end: int,
    ranks: Optional[Iterable[int]] = None,
    include_step: bool = True,
) -> Dict[int, List[Dict[str, Any]]]:
    """
    Collect micro-batches in an inclusive optimizer-step range, grouped by rank.

    Parameters
    ----------
    exposures_dir:
        Path to exposure logs directory.
    step_start, step_end:
        Inclusive step bounds. Example: 100..300 includes 100 and 300.
    ranks:
        Optional subset of ranks to include. If None, include all discovered ranks.
    include_step:
        If True, enrich each returned micro-batch dict with context fields:
        `step`, `timestamp`, `rank`, `world_size`.

    Returns
    -------
    Dict[int, List[Dict[str, Any]]]
        `{rank: [micro_batch_dict, ...]}`

    Shape intuition
    ---------------
    For this run:
    - exposure records are written every 100 optimizer steps
    - each record contains 6 micro-batches (because grad_accum_steps=6)
    So a 3-record window usually gives 18 micro-batches per rank.
    """
    if step_end < step_start:
        raise ValueError("step_end must be >= step_start")

    # Discover files once and choose which ranks to read.
    file_map = discover_exposure_files(exposures_dir)
    wanted = set(file_map.keys()) if ranks is None else set(int(r) for r in ranks)

    # Pre-create output keys for stable downstream use.
    out: Dict[int, List[Dict[str, Any]]] = {r: [] for r in sorted(wanted)}

    for rank, path in file_map.items():
        if rank not in wanted:
            continue

        for rec in iter_exposure_records(path):
            step = int(rec.get("step", -1))

            # Keep only requested step interval.
            if step < step_start or step > step_end:
                continue

            mbs = rec.get("micro_batches", [])
            if not isinstance(mbs, list):
                # Skip malformed records instead of crashing all results.
                continue

            if include_step:
                # Attach exposure-record context to each micro-batch item.
                for mb in mbs:
                    item = dict(mb)
                    item["step"] = step
                    item["timestamp"] = rec.get("timestamp")
                    item["rank"] = rec.get("rank")
                    item["world_size"] = rec.get("world_size")
                    out[rank].append(item)
            else:
                # Return raw micro-batch dicts exactly as stored in file.
                out[rank].extend(mbs)

    return out


def get_microbatches_by_rank_and_step(
    exposures_dir: Path,
    step_start: int,
    step_end: int,
    ranks: Optional[Iterable[int]] = None,
) -> Dict[int, Dict[int, List[Dict[str, Any]]]]:
    """
    Alternative organization for the same data:

      {rank: {step: [micro_batch_dict, ...]}}

    Use this when you care about per-step grouping (for example, comparing
    rank behavior step-by-step) rather than one flattened list per rank.
    """
    if step_end < step_start:
        raise ValueError("step_end must be >= step_start")

    file_map = discover_exposure_files(exposures_dir)
    wanted = set(file_map.keys()) if ranks is None else set(int(r) for r in ranks)

    out: Dict[int, Dict[int, List[Dict[str, Any]]]] = {r: {} for r in sorted(wanted)}

    for rank, path in file_map.items():
        if rank not in wanted:
            continue

        for rec in iter_exposure_records(path):
            step = int(rec.get("step", -1))
            if step < step_start or step > step_end:
                continue

            mbs = rec.get("micro_batches", [])
            if not isinstance(mbs, list):
                continue

            # One exposure record per step per rank in this run.
            out[rank][step] = mbs

    return out


## Mental Model: Packed Rows and Decoding

Each row in shard data has `seq_len + 1` tokens (here: `1025`):

- row[0] is BOS/EOS token id (50256)
- training input is `row[:-1]`
- training label is `row[1:]`

A micro-batch points to a contiguous chunk in a shard. To decode it:

1. memmap the shard (`uint16`)
2. slice token offsets `[start:end]`
3. reshape into `(-1, seq_len + 1)` rows
4. decode each row with GPT-2 tokenizer

That gives you the exact row-text the model consumed for those micro-batches.


In [ ]:
@lru_cache(maxsize=1)
def get_default_gpt2_tokenizer():
    """
    Load and cache the default GPT-2 tokenizer.

    Why cache:
    - `AutoTokenizer.from_pretrained("gpt2")` is relatively expensive.
    - We decode repeatedly, so caching avoids reloading vocab/merges each call.
    """
    return AutoTokenizer.from_pretrained("gpt2", use_fast=True)


def load_microbatch_rows(
    microbatch: Dict[str, Any],
    seq_len: int = 1024,
    dtype=np.uint16,
) -> np.ndarray:
    """
    Read one micro-batch directly from shard offsets and reshape into rows.

    Input expectations
    ------------------
    `microbatch` must contain:
    - `shard_path`: path to `train_*.bin` / `val_*.bin`
    - `start`: token offset (inclusive)
    - `end`: token offset (exclusive)

    Output
    ------
    np.ndarray
        Shape `(B, seq_len + 1)` where B is inferred from `end - start`.

    Important
    ---------
    Offsets are in *tokens*, not bytes. Because shard dtype is uint16,
    each token occupies 2 bytes on disk; numpy memmap handles that for us.
    """
    shard_path = Path(microbatch["shard_path"])
    start = int(microbatch["start"])
    end = int(microbatch["end"])

    if end <= start:
        raise ValueError(f"Invalid start/end offsets: start={start}, end={end}")

    # Every packed row has seq_len + 1 tokens.
    # Example: seq_len=1024 => 1025 tokens/row.
    row_tokens = int(seq_len) + 1

    # Memory-map file read-only; this does not load the whole shard into RAM.
    mm = np.memmap(shard_path, dtype=dtype, mode="r")

    # Slice token range for this micro-batch.
    chunk = np.asarray(mm[start:end], dtype=np.int64)

    # Safety check: token count must be whole number of rows.
    if chunk.size % row_tokens != 0:
        raise ValueError(
            f"Chunk size {chunk.size} is not divisible by row_tokens={row_tokens}. "
            f"Check seq_len and offsets."
        )

    # Infer B automatically and reshape to [B, row_tokens].
    return chunk.reshape(-1, row_tokens)


def decode_microbatch(
    microbatch: Dict[str, Any],
    seq_len: int = 1024,
    skip_special_tokens: bool = False,
    max_rows: Optional[int] = None,
    include_input_and_label: bool = False,
) -> Dict[str, Any]:
    """
    Decode one selected micro-batch using the default GPT-2 tokenizer.

    Parameters
    ----------
    microbatch:
        One micro-batch dict from exposure logs (or from helper outputs).
    seq_len:
        Training sequence length used to build rows; default 1024.
    skip_special_tokens:
        Passed to tokenizer decode.
        For this dataset, rows start with BOS/EOS token id 50256, so setting
        this True hides that token in decoded text.
    max_rows:
        If set, decode only first N rows from the micro-batch.
    include_input_and_label:
        If True, also produce decoded `input_ids` and `label_ids` views where:
        - input_ids = row[:-1]
        - label_ids = row[1:]

    Returns
    -------
    Dict[str, Any]
        {
          "microbatch_meta": ...,
          "num_rows_decoded": int,
          "decoded_rows": [
             {
               "row_index_in_microbatch": int,
               "token_ids": [...],
               "text": str,
               ... optional input/label fields ...
             },
             ...
          ]
        }
    """
    tok = get_default_gpt2_tokenizer()

    # Load token rows from the shard based on start/end offsets.
    rows = load_microbatch_rows(microbatch, seq_len=seq_len)

    # Optional truncation for quick inspection.
    if max_rows is not None:
        rows = rows[: int(max_rows)]

    decoded_rows: List[Dict[str, Any]] = []

    for i, row in enumerate(rows):
        row_ids = row.tolist()

        # Base decoded view for the full packed row.
        item: Dict[str, Any] = {
            "row_index_in_microbatch": i,
            "token_ids": row_ids,
            "text": tok.decode(row_ids, skip_special_tokens=skip_special_tokens),
        }

        if include_input_and_label:
            # Match the training objective split used by the loader.
            x_ids = row[:-1].tolist()
            y_ids = row[1:].tolist()

            item["input_ids"] = x_ids
            item["label_ids"] = y_ids
            item["input_text"] = tok.decode(x_ids, skip_special_tokens=skip_special_tokens)
            item["label_text"] = tok.decode(y_ids, skip_special_tokens=skip_special_tokens)

        decoded_rows.append(item)

    return {
        "microbatch_meta": {
            "step": microbatch.get("step"),
            "rank": microbatch.get("rank"),
            "shard_idx": microbatch.get("shard_idx"),
            "shard_path": microbatch.get("shard_path"),
            "block_idx": microbatch.get("block_idx"),
            "start": microbatch.get("start"),
            "end": microbatch.get("end"),
            "worker_id": microbatch.get("worker_id"),
            "num_workers": microbatch.get("num_workers"),
        },
        "num_rows_decoded": len(decoded_rows),
        "decoded_rows": decoded_rows,
    }


## Step-Based Decode Utilities

These helpers let you select arbitrary steps (set or range) and decode exactly what text was exposed during those steps.

In [ ]:
def _normalize_step_selector(
    steps: Optional[Iterable[int]] = None,
    step_start: Optional[int] = None,
    step_end: Optional[int] = None,
) -> tuple[Optional[set[int]], Optional[int], Optional[int]]:
    """
    Normalize step-selection inputs into one internal representation.

    You can select steps in exactly one of these ways:
    - explicit set/list: `steps=[100, 500, 1000]`
    - inclusive range: `step_start=100, step_end=1000`
    - all available steps: leave everything as None

    Returns
    -------
    (steps_set, range_start, range_end)
      - steps_set is a set[int] when explicit steps are used, else None
      - range_start/range_end are ints when range selection is used, else None
    """
    using_explicit_steps = steps is not None
    using_range = (step_start is not None) or (step_end is not None)

    # Keep the API unambiguous: either explicit steps OR range.
    if using_explicit_steps and using_range:
        raise ValueError("Use either `steps` OR (`step_start`, `step_end`), not both.")

    if using_explicit_steps:
        steps_set = {int(s) for s in steps}
        if not steps_set:
            raise ValueError("`steps` was provided but empty.")
        return steps_set, None, None

    if using_range:
        if step_start is None or step_end is None:
            raise ValueError("When using range selection, provide both step_start and step_end.")
        step_start = int(step_start)
        step_end = int(step_end)
        if step_end < step_start:
            raise ValueError("step_end must be >= step_start")
        return None, step_start, step_end

    # No selector provided: keep all steps.
    return None, None, None


def _step_is_selected(
    step: int,
    steps_set: Optional[set[int]],
    range_start: Optional[int],
    range_end: Optional[int],
) -> bool:
    """Return True if `step` passes the normalized selector."""
    if steps_set is not None:
        return step in steps_set
    if range_start is not None and range_end is not None:
        return range_start <= step <= range_end
    return True


def iter_microbatches_for_steps(
    exposures_dir: Path,
    steps: Optional[Iterable[int]] = None,
    step_start: Optional[int] = None,
    step_end: Optional[int] = None,
    ranks: Optional[Iterable[int]] = None,
) -> Iterable[Dict[str, Any]]:
    """
    Yield micro-batches whose record step matches the requested selector.

    This is the most useful low-level iterator for exposure analysis. It yields
    one dict per micro-batch, already enriched with step/rank context.

    Yielded keys include original micro-batch fields plus:
    - `step`, `timestamp`, `rank`, `world_size`
    - `microbatch_index_in_record` (position inside that record's list)
    """
    file_map = discover_exposure_files(exposures_dir)
    wanted = set(file_map.keys()) if ranks is None else {int(r) for r in ranks}

    steps_set, range_start, range_end = _normalize_step_selector(
        steps=steps,
        step_start=step_start,
        step_end=step_end,
    )

    # Iterate ranks deterministically for reproducible ordering.
    for rank in sorted(wanted):
        path = file_map.get(rank)
        if path is None:
            # If caller requested a rank not present on disk, skip quietly.
            continue

        for rec in iter_exposure_records(path):
            step = int(rec.get("step", -1))
            if not _step_is_selected(step, steps_set, range_start, range_end):
                continue

            mbs = rec.get("micro_batches", [])
            if not isinstance(mbs, list):
                continue

            for mb_idx, mb in enumerate(mbs):
                item = dict(mb)
                item["step"] = step
                item["timestamp"] = rec.get("timestamp")
                item["rank"] = int(rec.get("rank", rank))
                item["world_size"] = rec.get("world_size")
                item["microbatch_index_in_record"] = mb_idx
                yield item


def iter_decoded_rows_for_steps(
    exposures_dir: Path,
    steps: Optional[Iterable[int]] = None,
    step_start: Optional[int] = None,
    step_end: Optional[int] = None,
    ranks: Optional[Iterable[int]] = None,
    seq_len: int = 1024,
    skip_special_tokens: bool = False,
    max_microbatches: Optional[int] = None,
    max_rows_per_microbatch: Optional[int] = None,
    include_token_ids: bool = False,
    include_input_and_label: bool = False,
) -> Iterable[Dict[str, Any]]:
    """
    Stream decoded row-level exposure examples for selected steps.

    Why this exists:
    - It avoids loading everything into memory.
    - You can feed it directly into analysis pipelines.

    Each yielded row record contains text + provenance metadata, including
    rank, step, shard path, block index, and row position in micro-batch.
    """
    tok = get_default_gpt2_tokenizer()

    yielded_microbatches = 0

    for mb in iter_microbatches_for_steps(
        exposures_dir=exposures_dir,
        steps=steps,
        step_start=step_start,
        step_end=step_end,
        ranks=ranks,
    ):
        if max_microbatches is not None and yielded_microbatches >= int(max_microbatches):
            break

        rows = load_microbatch_rows(mb, seq_len=seq_len)
        if max_rows_per_microbatch is not None:
            rows = rows[: int(max_rows_per_microbatch)]

        for row_idx, row in enumerate(rows):
            row_ids = row.tolist()

            out: Dict[str, Any] = {
                "step": mb.get("step"),
                "rank": mb.get("rank"),
                "timestamp": mb.get("timestamp"),
                "shard_idx": mb.get("shard_idx"),
                "shard_path": mb.get("shard_path"),
                "block_idx": mb.get("block_idx"),
                "start": mb.get("start"),
                "end": mb.get("end"),
                "microbatch_index_in_record": mb.get("microbatch_index_in_record"),
                "row_index_in_microbatch": row_idx,
                "text": tok.decode(row_ids, skip_special_tokens=skip_special_tokens),
            }

            if include_token_ids:
                out["token_ids"] = row_ids

            if include_input_and_label:
                # Mirrors train-time next-token objective split.
                x_ids = row[:-1].tolist()
                y_ids = row[1:].tolist()
                out["input_ids"] = x_ids
                out["label_ids"] = y_ids
                out["input_text"] = tok.decode(x_ids, skip_special_tokens=skip_special_tokens)
                out["label_text"] = tok.decode(y_ids, skip_special_tokens=skip_special_tokens)

            yield out

        yielded_microbatches += 1


def decode_text_exposure_for_steps(
    exposures_dir: Path,
    steps: Optional[Iterable[int]] = None,
    step_start: Optional[int] = None,
    step_end: Optional[int] = None,
    ranks: Optional[Iterable[int]] = None,
    seq_len: int = 1024,
    skip_special_tokens: bool = False,
    max_microbatches: Optional[int] = None,
    max_rows_per_microbatch: Optional[int] = None,
    include_token_ids: bool = False,
    include_input_and_label: bool = False,
    group_by: str = "step",
) -> Dict[Any, Any]:
    """
    High-level convenience wrapper: decode exposure text and return grouped dict.

    group_by options
    ----------------
    - "step":      {step: [row_record, ...]}
    - "rank":      {rank: [row_record, ...]}
    - "step_rank": {step: {rank: [row_record, ...]}}

    Use this when you want immediate in-memory data structures for analysis.
    For larger extracts, prefer `iter_decoded_rows_for_steps` (streaming).
    """
    if group_by not in {"step", "rank", "step_rank"}:
        raise ValueError("group_by must be one of: 'step', 'rank', 'step_rank'")

    if group_by == "step_rank":
        out: Dict[int, Dict[int, List[Dict[str, Any]]]] = {}
        for rec in iter_decoded_rows_for_steps(
            exposures_dir=exposures_dir,
            steps=steps,
            step_start=step_start,
            step_end=step_end,
            ranks=ranks,
            seq_len=seq_len,
            skip_special_tokens=skip_special_tokens,
            max_microbatches=max_microbatches,
            max_rows_per_microbatch=max_rows_per_microbatch,
            include_token_ids=include_token_ids,
            include_input_and_label=include_input_and_label,
        ):
            step = int(rec["step"])
            rank = int(rec["rank"])
            out.setdefault(step, {}).setdefault(rank, []).append(rec)
        return out

    out_simple: Dict[int, List[Dict[str, Any]]] = {}
    for rec in iter_decoded_rows_for_steps(
        exposures_dir=exposures_dir,
        steps=steps,
        step_start=step_start,
        step_end=step_end,
        ranks=ranks,
        seq_len=seq_len,
        skip_special_tokens=skip_special_tokens,
        max_microbatches=max_microbatches,
        max_rows_per_microbatch=max_rows_per_microbatch,
        include_token_ids=include_token_ids,
        include_input_and_label=include_input_and_label,
    ):
        key = int(rec["step"]) if group_by == "step" else int(rec["rank"])
        out_simple.setdefault(key, []).append(rec)

    return out_simple


def extract_text_only_for_steps(
    exposures_dir: Path,
    steps: Optional[Iterable[int]] = None,
    step_start: Optional[int] = None,
    step_end: Optional[int] = None,
    ranks: Optional[Iterable[int]] = None,
    seq_len: int = 1024,
    skip_special_tokens: bool = False,
    max_microbatches: Optional[int] = None,
    max_rows_per_microbatch: Optional[int] = None,
) -> Dict[int, List[str]]:
    """
    Text-only helper for quick extraction.

    Returns
    -------
    Dict[int, List[str]]
        `{step: [decoded_text_row_0, decoded_text_row_1, ...]}`

    This is a compact API when you only care about strings, not metadata.
    """
    by_step = decode_text_exposure_for_steps(
        exposures_dir=exposures_dir,
        steps=steps,
        step_start=step_start,
        step_end=step_end,
        ranks=ranks,
        seq_len=seq_len,
        skip_special_tokens=skip_special_tokens,
        max_microbatches=max_microbatches,
        max_rows_per_microbatch=max_rows_per_microbatch,
        include_token_ids=False,
        include_input_and_label=False,
        group_by="step",
    )

    return {step: [r["text"] for r in records] for step, records in by_step.items()}


In [ ]:
# Example 1: collect micro-batches for a step range.
step_start = 100
step_end = 200

by_rank = get_microbatches_by_rank(
    exposures_dir=EXPOSURES_DIR,
    step_start=step_start,
    step_end=step_end,
)

summary = {rank: len(mbs) for rank, mbs in by_rank.items()}
summary


In [ ]:
# Example 2: get nested view {rank: {step: [micro_batches...]}}
by_rank_step = get_microbatches_by_rank_and_step(
    exposures_dir=EXPOSURES_DIR,
    step_start=0,
    step_end=3000,
)

# Peek rank 1 at step 100
by_rank_step[1][100][0]


In [ ]:
# Example 3: decode one selected micro-batch.
rank = 1
microbatch_index = 1

selected_mb = by_rank[rank][microbatch_index]
decoded = decode_microbatch(
    selected_mb,
    seq_len=1024,
    skip_special_tokens=False,
    max_rows=2,  # decode only first 2 rows for quick preview
    include_input_and_label=False,
)

decoded["microbatch_meta"], decoded["decoded_rows"][1]["text"][:600]


## Step-Based Decode Examples

In [ ]:
# Example 4: decode exposure text for an arbitrary set of steps.
selected_steps = [100, 500, 1000]

exposed_by_step = decode_text_exposure_for_steps(
    exposures_dir=EXPOSURES_DIR,
    steps=selected_steps,
    ranks=[0, 1],                 # optional rank subset
    seq_len=1024,
    skip_special_tokens=False,
    max_rows_per_microbatch=1,    # keep sample small for quick iteration
    group_by="step",
)

# How many decoded rows per selected step?
{step: len(rows) for step, rows in exposed_by_step.items()}


In [ ]:
# Example 5: extract text-only payload for downstream analysis.
text_only = extract_text_only_for_steps(
    exposures_dir=EXPOSURES_DIR,
    step_start=0,
    step_end=10000,
    ranks=[0,1,2,3,4,5,6,7],

)

first_step = sorted(text_only.keys())[0]
first_step, text_only[first_step][3][:500]


In [ ]:
text_only.keys()

In [ ]:


# import argparse
# import json
# import re
# from typing import Dict, List, Tuple, Optional, Iterable


# ###############################################################################
# # 1) Verb patterns
# ###############################################################################

# VERB_PATTERNS: Dict[str, str] = {
#     "pour":    r"\b(pour|pours|poured|pouring)\b",
#     "spill":   r"\b(spill|spills|spilled|spilling)\b",
#     "squeeze": r"\b(squeeze|squeezes|squeezed|squeezing)\b",
#     "stir":    r"\b(stir|stirs|stirred|stirring)\b",
#     "ripple":  r"\b(ripple|ripples|rippled|rippling)\b",
# }

# verb_re: Dict[str, re.Pattern] = {k: re.compile(v, re.I) for k, v in VERB_PATTERNS.items()}
# any_verb_re: re.Pattern = re.compile("|".join(VERB_PATTERNS.values()), re.I)


# ###############################################################################
# # 2) Literal-frame cues (broad physical cues; not “material dynamics only”)
# ###############################################################################

# POUR_CUES = re.compile(
#     r"\b(into|onto|from|out of|in|over|through)\b"
#     r"|\b(cup|glass|bottle|pitcher|jug|mug|bowl|pan|pot|sink|bucket|mold|molds|cylinder|beaker|flask|container)\b"
#     r"|\b(liquid|water|oil|milk|juice|tea|coffee|hcl|acid)\b",
#     re.I
# )

# SPILL_CUES = re.compile(
#     r"\b(onto|into|over|across|down|from)\b"
#     r"|\b(table|floor|ground|counter|surface|sink|tray)\b"
#     r"|\b(liquid|water|oil|milk|juice|tea|coffee|acid)\b",
#     re.I
# )

# STIR_CUES = re.compile(
#     r"\b(in|into)\b"
#     r"|\b(bowl|pot|pan|mug|cup)\b"
#     r"|\b(mixture|soup|stew|sauce|batter|eggs?|cream|coffee|tea)\b"
#     r"|\b(spoon|whisk|ladle|fork|spatula)\b",
#     re.I
# )

# SQUEEZE_CUES = re.compile(
#     r"\b(with|in|between)\b"
#     r"|\b(hand|hands|fingers|grip|pressure|palm)\b"
#     r"|\b(tube|toothpaste|sponge|lemon|lime|cloth|towel|rag)\b"
#     r"|\b(juice|drops?)\b",
#     re.I
# )

# # Ripple: avoid generic "fabric" cue (causes spacetime false positives).
# # Treat ripples as:
# #   - water/surface waves OR
# #   - cloth/flag/curtain rippling in wind
# RIPPLE_WATER_CUES = re.compile(
#     r"\b(surface|water|lake|pond|sea|ocean|pool|puddle)\b"
#     r"|\b(wave|waves|wind|breeze|pebble|stone|drop|raindrop)\b"
#     r"|\b(across|over|along|outward)\b",
#     re.I
# )

# RIPPLE_CLOTH_CUES = re.compile(
#     r"\b(flag|curtain|cloth|fabric|dress|shirt|blanket)\b.*\b(wind|breeze|air)\b"
#     r"|\b(wind|breeze|air)\b.*\b(flag|curtain|cloth|fabric|dress|shirt|blanket)\b",
#     re.I
# )


# ###############################################################################
# # 3) Metaphor / idiom / noun-trap filters (targeted, conservative)
# ###############################################################################

# # Hard idioms (always reject)
# HARD_IDIOMS = [
#     r"\bstir up\b",
#     r"\bspill the beans\b",
#     r"\bsqueeze (in|out)\b",
#     r"\bripple effect\b",
#     r"\bripples?\s+in\s+the\s+fabric\s+of\s+spacetime\b",
# ]
# hard_idiom_re = re.compile("|".join(HARD_IDIOMS), re.I)

# # Noun-phrase traps (not an event frame)
# NOUN_TRAPS = [
#     r"\boil\s+spills\b",  # usually plural noun
#     r"\bripples?\s+in\s+the\s+fabric\s+of\s+spacetime\b",
# ]
# noun_trap_re = re.compile("|".join(NOUN_TRAPS), re.I)

# # Abstract cue words (used only near the verb)
# ABSTRACT_CUES = re.compile(
#     r"\b(economy|market|stocks?|policy|election|debate|controversy|rumor|news|funding|capital|profit|profits|margins"
#     r"|strategy|plan|timeline|deadline|performance|metrics"
#     r"|data|code|file|program|catalogs?|observations?|findings|reports|documents?)\b",
#     re.I
# )

# # Optional out-of-domain science/astro filter
# ASTRO_CUES = re.compile(
#     r"\b(nobel|einstein|spacetime|gravitational|waves?|telescope|hubble|comet|asteroid|galaxy|cosmic|physics)\b",
#     re.I
# )

# # Pour idioms are tricky because "water was pouring in" can be literal.
# # So we reject "pouring in/into" only if it looks like people/info/abstract stuff.
# POUR_IDIOM_CUES = re.compile(
#     r"\b(observations?|findings|reports|news|emails?|messages?|data|applications?|requests?|questions?)\b"
#     r"|\b(miners|people|crowds?|tourists|immigrants|students|workers|fans)\b"
#     r"|\b(california|city|country|state|town|market|economy)\b",
#     re.I
# )

# # "poured over" can be literal ("poured over pancakes") or abstract ("poured over the catalogs").
# # Reject only when the object is abstract.
# POUR_OVER_ABSTRACT = re.compile(
#     r"\bpour(ed|ing)?\s+over\b.*\b(data|code|catalogs?|documents?|files?|reports?)\b",
#     re.I
# )


# ###############################################################################
# # 4) Basic text helpers
# ###############################################################################

# PRONOUN_RE = re.compile(r"\b(it|they|this|that|these|those)\b", re.I)

# def clean_text(s: str) -> str:
#     s = s.replace("\u00a0", " ")
#     s = re.sub(r"\s+", " ", s).strip()
#     return s

# def split_sentences(text: str) -> List[str]:
#     """
#     Simple sentence splitter (good enough for mining).
#     """
#     text = text.replace("<|endoftext|>", " ").strip()
#     text = re.sub(r"\s+", " ", text)
#     sents = re.split(r"(?<=[\.\?\!])\s+", text)
#     return [s.strip() for s in sents if s and s.strip()]

# def abstract_near_match(sent: str, m: re.Match, radius: int = 70) -> bool:
#     span = sent[max(0, m.start()-radius):min(len(sent), m.end()+radius)]
#     return bool(ABSTRACT_CUES.search(span))

# def detect_verbs(sent: str) -> List[str]:
#     return [k for k, r in verb_re.items() if r.search(sent)]

# def cue_ok(sent: str, v: str) -> bool:
#     if v == "pour":
#         return bool(POUR_CUES.search(sent))
#     if v == "spill":
#         return bool(SPILL_CUES.search(sent))
#     if v == "stir":
#         return bool(STIR_CUES.search(sent))
#     if v == "squeeze":
#         return bool(SQUEEZE_CUES.search(sent))
#     if v == "ripple":
#         return bool(RIPPLE_WATER_CUES.search(sent) or RIPPLE_CLOTH_CUES.search(sent))
#     return False

# def looks_like_pour_idiom(sent: str) -> bool:
#     """
#     Reject pour(pouring) in/into if it looks like people/info/abstract uses.
#     Keep literal cases like "water was pouring in".
#     """
#     s = sent.lower()
#     if re.search(r"\b(pouring|poured|pour)\s+in\b", s) or re.search(r"\b(pouring|poured|pour)\s+into\b", s):
#         if POUR_IDIOM_CUES.search(sent):
#             return True
#     return False


# ###############################################################################
# # 5) Core literal candidate filter (broad)
# ###############################################################################

# def is_literal_candidate(
#     sent: str,
#     require_cue: bool = True,
#     reject_astro: bool = True,
# ) -> Tuple[bool, List[str]]:
#     """
#     Broad literalness filter:
#       - must contain at least one target verb
#       - reject hard idioms/noun traps
#       - reject abstract cues near the verb
#       - verb-specific cue requirement (optional)
#       - optional: reject astrophysics/cosmology contexts
#     Returns: (keep?, [verb_tags])
#     """
#     sent = clean_text(sent)
#     if not sent:
#         return False, []

#     if hard_idiom_re.search(sent):
#         return False, []
#     if noun_trap_re.search(sent):
#         return False, []
#     if POUR_OVER_ABSTRACT.search(sent):
#         return False, []
#     if looks_like_pour_idiom(sent):
#         return False, []

#     m = any_verb_re.search(sent)
#     if not m:
#         return False, []

#     if abstract_near_match(sent, m):
#         return False, []

#     if reject_astro and ASTRO_CUES.search(sent):
#         return False, []

#     verbs = detect_verbs(sent)
#     if require_cue:
#         for v in verbs:
#             if not cue_ok(sent, v):
#                 return False, []
#     return True, verbs


# ###############################################################################
# # 6) Mining: single sentences
# ###############################################################################

# def mine_literal_sentences(
#     text_sequences: List[str],
#     min_len_chars: int = 25,
#     max_len_chars: int = 260,
#     require_cue: bool = True,
#     reject_astro: bool = True,
# ) -> List[Dict]:
#     out: List[Dict] = []
#     for seq in text_sequences:
#         for sent in split_sentences(seq):
#             sent = clean_text(sent)
#             if not (min_len_chars <= len(sent) <= max_len_chars):
#                 continue
#             keep, verbs = is_literal_candidate(sent, require_cue=require_cue, reject_astro=reject_astro)
#             if not keep:
#                 continue
#             out.append({"sent": sent, "verbs": verbs})
#     return out

In [ ]:
import argparse
import json
import re
from typing import Any, Dict, List, Tuple, Optional, Iterable


###############################################################################
# 1) Verb patterns
###############################################################################

# Keep full set for later, but only mine ACTIVE_VERBS below.
VERB_PATTERNS: Dict[str, str] = {
    "pour":    r"\b(pour|pours|poured|pouring)\b",
    "spill":   r"\b(spill|spills|spilled|spilling)\b",
    "squeeze": r"\b(squeeze|squeezes|squeezed|squeezing)\b",
    "stir":    r"\b(stir|stirs|stirred|stirring)\b",
    "ripple":  r"\b(ripple|ripples|rippled|rippling)\b",
    # NEW
    "wrinkle": r"\b(wrinkle|wrinkles|wrinkled|wrinkling)\b",
    "hang":    r"\b(hang|hangs|hung|hanging)\b",
}

# NEW: prioritize these; everything else is "frozen"
ACTIVE_VERBS = {"stir", "wrinkle", "hang", "squeeze"}

ACTIVE_VERB_PATTERNS: Dict[str, str] = {k: v for k, v in VERB_PATTERNS.items() if k in ACTIVE_VERBS}
verb_re: Dict[str, re.Pattern] = {k: re.compile(v, re.I) for k, v in ACTIVE_VERB_PATTERNS.items()}
any_verb_re: re.Pattern = re.compile("|".join(ACTIVE_VERB_PATTERNS.values()), re.I)

# NEW: only apply pour-specific idiom filters if you're actually mining pour
ENABLE_POUR_FILTERS = "pour" in ACTIVE_VERBS


###############################################################################
# 2) Literal-frame cues (broad physical cues; not “material dynamics only”)
###############################################################################

POUR_CUES = re.compile(
    r"\b(into|onto|from|out of|in|over|through)\b"
    r"|\b(cup|glass|bottle|pitcher|jug|mug|bowl|pan|pot|sink|bucket|mold|molds|cylinder|beaker|flask|container)\b"
    r"|\b(liquid|water|oil|milk|juice|tea|coffee|hcl|acid)\b",
    re.I
)

SPILL_CUES = re.compile(
    r"\b(onto|into|over|across|down|from)\b"
    r"|\b(table|floor|ground|counter|surface|sink|tray)\b"
    r"|\b(liquid|water|oil|milk|juice|tea|coffee|acid)\b",
    re.I
)

STIR_CUES = re.compile(
    r"\b(in|into)\b"
    r"|\b(bowl|pot|pan|mug|cup)\b"
    r"|\b(mixture|soup|stew|sauce|batter|eggs?|cream|coffee|tea)\b"
    r"|\b(spoon|whisk|ladle|fork|spatula)\b"
    # OPTIONAL: add common recipe outcomes (keeps recall high without much noise)
    r"|\b(until|smooth|well|combined|dissolved)\b",
    re.I
)

SQUEEZE_CUES = re.compile(
    r"\b(with|in|between)\b"
    r"|\b(hand|hands|fingers|grip|pressure|palm)\b"
    r"|\b(tube|toothpaste|sponge|lemon|lime|cloth|towel|rag)\b"
    r"|\b(juice|drops?)\b",
    re.I
)

# NEW: wrinkle cues (press/crease/laundry/fabric/paper)
WRINKLE_CUES = re.compile(
    r"\b(press|pressed|pressing|crease|creased|creasing|crumple|crumpled|crumpling|fold|folded|folding|iron|ironing)\b"
    r"|\b(shirt|cloth|fabric|cotton|linen|paper|napkin|sheet|towel|dress|curtain)\b"
    r"|\b(laundry|dryer|wrinkle[- ]free)\b",
    re.I
)

# NEW: hang cues (support + gravity frames)
HANG_CUES = re.compile(
    r"\b(on|from|over|across)\b"
    r"|\b(hook|nail|rod|hanger|line|clothesline|rack|peg|branch|rail)\b"
    r"|\b(shirt|coat|towel|curtain|painting|picture|frame|laundry|clothes)\b"
    r"|\b(dry|drying|air[- ]dry)\b",
    re.I
)

# Ripple cues unchanged (but "ripple" is frozen unless you add it to ACTIVE_VERBS)
RIPPLE_WATER_CUES = re.compile(
    r"\b(surface|water|lake|pond|sea|ocean|pool|puddle)\b"
    r"|\b(wave|waves|wind|breeze|pebble|stone|drop|raindrop)\b"
    r"|\b(across|over|along|outward)\b",
    re.I
)

RIPPLE_CLOTH_CUES = re.compile(
    r"\b(flag|curtain|cloth|fabric|dress|shirt|blanket)\b.*\b(wind|breeze|air)\b"
    r"|\b(wind|breeze|air)\b.*\b(flag|curtain|cloth|fabric|dress|shirt|blanket)\b",
    re.I
)


###############################################################################
# 3) Metaphor / idiom / noun-trap filters (targeted, conservative)
###############################################################################

# Hard idioms (always reject)
HARD_IDIOMS = [
    r"\bstir up\b",
    r"\bspill the beans\b",

    # keep the narrow squeeze idiom filters
    r"\bsqueeze\s+in\b.*\b(time|schedule|meeting|appointment|deadline|minutes?|hours?)\b",
    r"\bsqueeze\s+out\b.*\b(profit|profits|margin|margins|cost|costs|efficien(?:cy|cies)|advantage|value)\b",

    # new: hang idioms
    r"\bhang\s+out\b",
    r"\bhang\s+on\b",
    r"\bhang\s+onto\b",

    # new: stir metaphor
    r"\bbegan\s+to\s+stir\b",
    r"\bstirring\s+tune\b",

    # new: squeeze quantitative
    r"\bsqueeze\s+as\s+much\s+as\b",

    r"\bripple effect\b",
    r"\bripples?\s+in\s+the\s+fabric\s+of\s+spacetime\b",
]

hard_idiom_re = re.compile("|".join(HARD_IDIOMS), re.I)

NOUN_TRAPS = [
    r"\boil\s+spills\b",
    r"\bripples?\s+in\s+the\s+fabric\s+of\s+spacetime\b",
]
noun_trap_re = re.compile("|".join(NOUN_TRAPS), re.I)

ABSTRACT_CUES = re.compile(
    r"\b(economy|market|stocks?|policy|election|debate|controversy|rumor|news|funding|capital|profit|profits|margins"
    r"|strategy|plan|timeline|deadline|performance|metrics"
    r"|data|code|file|program|catalogs?|observations?|findings|reports|documents?)\b",
    re.I
)

ASTRO_CUES = re.compile(
    r"\b(nobel|einstein|spacetime|gravitational|waves?|telescope|hubble|comet|asteroid|galaxy|cosmic|physics)\b",
    re.I
)

POUR_IDIOM_CUES = re.compile(
    r"\b(observations?|findings|reports|news|emails?|messages?|data|applications?|requests?|questions?)\b"
    r"|\b(miners|people|crowds?|tourists|immigrants|students|workers|fans)\b"
    r"|\b(california|city|country|state|town|market|economy)\b",
    re.I
)

POUR_OVER_ABSTRACT = re.compile(
    r"\bpour(ed|ing)?\s+over\b.*\b(data|code|catalogs?|documents?|files?|reports?)\b",
    re.I
)


###############################################################################
# 4) Basic text helpers
###############################################################################

PRONOUN_RE = re.compile(r"\b(it|they|this|that|these|those)\b", re.I)

def clean_text(s: str) -> str:
    s = s.replace("\u00a0", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def split_sentences(text: str) -> List[str]:
    text = text.replace("<|endoftext|>", " ").strip()
    text = re.sub(r"\s+", " ", text)
    sents = re.split(r"(?<=[\.\?\!])\s+", text)
    return [s.strip() for s in sents if s and s.strip()]

def abstract_near_match(sent: str, m: re.Match, radius: int = 70) -> bool:
    span = sent[max(0, m.start()-radius):min(len(sent), m.end()+radius)]
    return bool(ABSTRACT_CUES.search(span))

def detect_verbs(sent: str) -> List[str]:
    # Only detect ACTIVE verbs (frozen concepts won't appear here)
    return [k for k, r in verb_re.items() if r.search(sent)]

def cue_ok(sent: str, v: str) -> bool:
    if v == "pour":
        return bool(POUR_CUES.search(sent))
    if v == "spill":
        return bool(SPILL_CUES.search(sent))
    if v == "stir":
        return bool(STIR_CUES.search(sent))
    if v == "squeeze":
        return bool(SQUEEZE_CUES.search(sent))
    if v == "wrinkle":
        return bool(WRINKLE_CUES.search(sent))
    if v == "hang":
        return bool(HANG_CUES.search(sent))
    if v == "ripple":
        return bool(RIPPLE_WATER_CUES.search(sent) or RIPPLE_CLOTH_CUES.search(sent))
    return False

def looks_like_pour_idiom(sent: str) -> bool:
    s = sent.lower()
    if re.search(r"\b(pouring|poured|pour)\s+in\b", s) or re.search(r"\b(pouring|poured|pour)\s+into\b", s):
        if POUR_IDIOM_CUES.search(sent):
            return True
    return False


###############################################################################
# 5) Core literal candidate filter (broad)
###############################################################################

def is_literal_candidate(
    sent: str,
    require_cue: bool = True,
    reject_astro: bool = True,
) -> Tuple[bool, List[str]]:
    sent = clean_text(sent)
    if not sent:
        return False, []

    if hard_idiom_re.search(sent):
        return False, []
    if noun_trap_re.search(sent):
        return False, []

    # Only apply pour-specific filters if pour is active (otherwise it can block good stir/hang/etc. sentences)
    if ENABLE_POUR_FILTERS:
        if POUR_OVER_ABSTRACT.search(sent):
            return False, []
        if looks_like_pour_idiom(sent):
            return False, []

    m = any_verb_re.search(sent)
    if not m:
        return False, []

    if abstract_near_match(sent, m):
        return False, []

    if reject_astro and ASTRO_CUES.search(sent):
        return False, []

    verbs = detect_verbs(sent)
    if require_cue:
        for v in verbs:
            if not cue_ok(sent, v):
                return False, []
    return True, verbs


###############################################################################
# 6) Mining: single sentences
###############################################################################

def mine_literal_sentences(
    text_sequences: List[str],
    min_len_chars: int = 25,
    max_len_chars: int = 260,
    require_cue: bool = True,
    reject_astro: bool = True,
) -> List[Dict]:
    out: List[Dict] = []
    for seq in text_sequences:
        for sent in split_sentences(seq):
            sent = clean_text(sent)
            if not (min_len_chars <= len(sent) <= max_len_chars):
                continue
            keep, verbs = is_literal_candidate(sent, require_cue=require_cue, reject_astro=reject_astro)
            if not keep:
                continue
            out.append({"sent": sent, "verbs": verbs})
    return out

###############################################################################
# 7) Swappable key noun + literal-frame post-filters
###############################################################################

LIQUIDS = [
    "water","oil","milk","juice","coffee","tea","vinegar","paint","ink",
    "hcl","acid","syrup","honey","soap","detergent"
]
GRANULARS = [
    "sand","salt","sugar","flour","rice","gravel","soil","beans","pepper","powder"
]
PASTES = [
    "mud","clay","dough","putty","foam","paste","batter","cream","gel","slime","wax"
]
SHEETS = [
    "paper","cloth","fabric","towel","blanket","curtain","sheet","rag","napkin","film"
]
BRITTLE = [
    "glass","ceramic","porcelain","ice","crystal"
]
POLYMERS = [
    "plastic","acrylic","polycarbonate","rubber","silicone","vinyl"
]

MATERIAL_LEXICON = set(LIQUIDS + GRANULARS + PASTES + SHEETS + BRITTLE + POLYMERS)

DEFAULT_SWAP_GROUP = {
    "liquid": "granular",
    "granular": "liquid",
    "paste": "granular",
    "sheet": "brittle",   # requested change
    "brittle": "polymer",
    "polymer": "brittle",
}

GROUPS = {
    "liquid": set(LIQUIDS),
    "granular": set(GRANULARS),
    "paste": set(PASTES),
    "sheet": set(SHEETS),
    "brittle": set(BRITTLE),
    "polymer": set(POLYMERS),
}


def get_text(item: Dict[str, Any]) -> str:
    if "C1" in item and "T1" in item:
        return f"{item.get('C1','')} {item.get('T1','')}".strip()
    if "sent" in item:
        return item["sent"]
    parts = [v for v in item.values() if isinstance(v, str)]
    return " ".join(parts)


def get_context_text(item: Dict[str, Any]) -> str:
    if "C1" in item:
        return item["C1"]
    if "sent" in item:
        return item["sent"]
    return get_text(item)


def get_target_text(item: Dict[str, Any]) -> str:
    if "T1" in item:
        return item["T1"]
    return get_text(item)


def detect_material_group(word: str) -> Optional[str]:
    w = word.lower()
    for g, vocab in GROUPS.items():
        if w in vocab:
            return g
    return None


def find_swappable_key_noun(text: str) -> Optional[Dict[str, str]]:
    tokens = re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", (text or "").lower())
    for t in tokens:
        if t in MATERIAL_LEXICON:
            g = detect_material_group(t)
            if not g:
                continue
            return {
                "noun": t,
                "group": g,
                "suggested_swap_group": DEFAULT_SWAP_GROUP.get(g, ""),
            }
    return None


def has_swappable_key_noun(item: Dict[str, Any], require_in_context: bool = True) -> Tuple[bool, Dict[str, str]]:
    ctx = get_context_text(item) if require_in_context else get_text(item)
    hit = find_swappable_key_noun(ctx)
    if hit:
        return True, hit

    if require_in_context:
        hit2 = find_swappable_key_noun(get_target_text(item))
        if hit2:
            return True, hit2

    return False, {}


def is_literal_physical_frame(item: Dict[str, Any], require_cue: bool = True) -> Tuple[bool, str]:
    text = get_text(item)

    if hard_idiom_re.search(text):
        return False, "idiom_blacklist"

    m = any_verb_re.search(text)
    if not m:
        return False, "no_target_verb"

    if abstract_near_match(text, m):
        return False, "abstract_near_verb"

    if ENABLE_POUR_FILTERS and looks_like_pour_idiom(text):
        return False, "pouring_in/into_idiom"

    verbs = item.get("verbs")
    if not verbs:
        verbs = detect_verbs(text)
    verbs = [v for v in verbs if v in ACTIVE_VERBS]
    if not verbs:
        return False, "no_active_verbs"

    if require_cue:
        for v in verbs:
            if not cue_ok(text, v):
                return False, f"{v}_missing_physical_cue"

    return True, ""


def key_near_any_active_verb(text: str, key_noun: str, radius: int = 60) -> bool:
    if not key_noun:
        return False
    key_re = re.compile(r"\b" + re.escape(key_noun) + r"\b", re.I)
    for m in any_verb_re.finditer(text):
        span = text[max(0, m.start()-radius):min(len(text), m.end()+radius)]
        if key_re.search(span):
            return True
    return False


def filter_items(
    items: List[Dict[str, Any]],
    require_swappable_noun: bool = True,
    require_noun_in_context: bool = True,
    require_literal_frame: bool = True,
    require_cue_for_literal: bool = True,
    attach_metadata: bool = True,
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    kept, rejected = [], []

    for it in items:
        it2 = dict(it)

        if require_swappable_noun:
            ok_noun, noun_info = has_swappable_key_noun(it2, require_in_context=require_noun_in_context)
            if not ok_noun:
                if attach_metadata:
                    it2["reject_reason"] = "no_swappable_key_noun"
                rejected.append(it2)
                continue

            if attach_metadata:
                it2["key_noun"] = noun_info.get("noun", "")
                it2["key_noun_group"] = noun_info.get("group", "")
                it2["suggested_swap_group"] = noun_info.get("suggested_swap_group", "")

        if require_literal_frame and attach_metadata:
            if not key_near_any_active_verb(get_text(it2), it2.get("key_noun", ""), radius=60):
                it2["reject_reason"] = "key_noun_not_near_verb"
                rejected.append(it2)
                continue

        if require_literal_frame:
            ok_lit, reason = is_literal_physical_frame(it2, require_cue=require_cue_for_literal)
            if not ok_lit:
                if attach_metadata:
                    it2["reject_reason"] = reason
                rejected.append(it2)
                continue

        kept.append(it2)

    return kept, rejected


def filter_for_contrast_pairs(
    items: List[Dict[str, Any]],
    allowed_verbs: Optional[List[str]] = None,
    disallow_multi_verb: bool = True,
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    if allowed_verbs is None:
        allowed_verbs = ["stir", "wrinkle", "hang", "squeeze"]
    allowed = set(allowed_verbs)

    kept, rejected = [], []
    for it in items:
        it2 = dict(it)
        verbs = it2.get("verbs") or []
        verbs = [v for v in verbs if v in allowed]

        if not verbs:
            it2["reject_reason"] = "no_allowed_verbs"
            rejected.append(it2)
            continue

        if disallow_multi_verb and len(verbs) != 1:
            it2["reject_reason"] = "multi_verb_item"
            rejected.append(it2)
            continue

        it2["verbs"] = verbs
        kept.append(it2)

    return kept, rejected


In [ ]:

###############################################################################
# 7) Mining: (C1, T1) pairs
###############################################################################

def mine_C1_T1_pairs_adjacent(
    text_sequences: List[str],
    min_c1_chars: int = 20,
    min_t1_chars: int = 10,
    max_total_chars: int = 340,
    require_pronoun_in_T1: bool = True,
    require_cue: bool = True,
    reject_astro: bool = True,
) -> List[Dict]:
    """
    Adjacent-sentence mining:
      C1 = sentence[i]
      T1 = sentence[i+1]  (T1 must contain literal verb usage)
    """
    out: List[Dict] = []
    for seq in text_sequences:
        sents = split_sentences(seq)
        for i in range(len(sents) - 1):
            C1 = clean_text(sents[i])
            T1 = clean_text(sents[i + 1])

            if len(C1) < min_c1_chars or len(T1) < min_t1_chars:
                continue
            if len(C1) + 1 + len(T1) > max_total_chars:
                continue

            keep_t1, verbs = is_literal_candidate(T1, require_cue=require_cue, reject_astro=reject_astro)
            if not keep_t1:
                continue

            # Avoid obvious idioms in context too (light touch)
            if hard_idiom_re.search(C1) or noun_trap_re.search(C1) or POUR_OVER_ABSTRACT.search(C1):
                continue
            if looks_like_pour_idiom(C1):
                continue

            if require_pronoun_in_T1 and not PRONOUN_RE.search(T1):
                continue

            out.append({"C1": C1, "T1": T1, "verbs": verbs, "mode": "adjacent"})
    return out


def mine_C1_T1_pairs_split(
    text_sequences: List[str],
    min_c1_chars: int = 20,
    min_t1_chars: int = 10,
    max_total_chars: int = 340,
    require_cue: bool = True,
    reject_astro: bool = True,
) -> List[Dict]:
    """
    Split-sentence mining:
      Take a single sentence that is a literal candidate and split at the first verb match:
        C1 = prefix before verb
        T1 = verb+rest
    """
    out: List[Dict] = []
    for seq in text_sequences:
        for sent in split_sentences(seq):
            sent = clean_text(sent)
            if len(sent) > max_total_chars:
                continue

            keep, verbs = is_literal_candidate(sent, require_cue=require_cue, reject_astro=reject_astro)
            if not keep:
                continue

            m = any_verb_re.search(sent)
            if not m:
                continue

            C1 = sent[:m.start()].rstrip()
            T1 = sent[m.start():].lstrip()

            if len(C1) < min_c1_chars or len(T1) < min_t1_chars:
                continue

            out.append({"C1": C1, "T1": T1, "verbs": verbs, "mode": "split"})
    return out


def mine_C1_T1_pairs(
    text_sequences: List[str],
    mode: str = "both",  # "adjacent" | "split" | "both"
    **kwargs
) -> List[Dict]:
    out: List[Dict] = []
    if mode in ("adjacent", "both"):
        out.extend(mine_C1_T1_pairs_adjacent(text_sequences, **kwargs))
    if mode in ("split", "both"):
        out.extend(mine_C1_T1_pairs_split(text_sequences, **kwargs))
    return out



In [ ]:
# we join all rows from every step into one big list, and then mine candidates from that.
all_rows = []
for step, rows in text_only.items():
    all_rows.extend(rows)
# mined_text = mine_candidates(all_rows)
mined_text = mine_literal_sentences(all_rows, require_cue=True)

In [ ]:
pairs = mine_C1_T1_pairs(all_rows, require_cue=True)

In [ ]:
len(pairs)

In [ ]:
# kept, rejected = filter_items(
#     pairs,
#     require_swappable_noun=True,
#     require_noun_in_context=True,
#     require_literal_frame=True,
#     require_cue_for_literal=True,
#     attach_metadata=True,
# )

# kept2, rejected2 = filter_for_contrast_pairs(
#     kept,
#     allowed_verbs=["pour","spill","squeeze","stir","ripple"],
#     disallow_multi_verb=True,   # strongly recommended initially
# )
# pairs = mine_C1_T1_pairs(all_rows, require_cue=True)

kept, rejected = filter_items(
    pairs,
    require_swappable_noun=True,
    require_noun_in_context=True,
    require_literal_frame=True,
    require_cue_for_literal=True,
    attach_metadata=True,
)

kept2, rejected2 = filter_for_contrast_pairs(
    kept,
    allowed_verbs=["stir", "wrinkle", "hang", "squeeze"],  # updated priorities
    disallow_multi_verb=True,
)

In [ ]:
len(kept)

In [ ]:
kept2

In [ ]:
print(len(kept2))

In [ ]:
# OpenAI API setup
import os
from openai import OpenAI

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY in your environment before running this notebook.")

client = OpenAI()
print("OpenAI client initialized")


In [ ]:
prompt = """You are generating EWoK-style counterfactual minimal pairs for MATERIAL DYNAMICS.
(You do NOT need to know anything about EWoK—just follow the constraints below.)

INPUT: Each item has C1_RAW (mined context) and T1_RAW (mined continuation).
Your output must be (C1, C2, T1, T2) where C1/C2 contrast in material class (A) and T1/T2 contrast in dynamics or consequence (D) with strong crossed plausibility.

LATENT VIEW: z = (E, A, D, R, S)
- E: entities/objects mentioned
- A: material class/state of ONE key object/substance
- D: a physical affordance/action/outcome dependent on A
- R,S: relations/internal states; keep stable unless explicitly present

========================
STRUCTURE TARGET (PREFERRED; REJECT IF NOT POSSIBLE CLEANLY)
========================
Prefer outputs where:
- C1 is exactly ONE sentence.
- C2 is exactly ONE sentence.
- T1 is exactly ONE sentence.
- T2 is exactly ONE sentence.
Do not split or merge sentences. If you cannot make this true without heavy paraphrase, output REJECT.

========================
CRITICAL FIX #1: RESEGMENT BEFORE EDITING
========================
Your mined C1_RAW may be unhelpful or not contain the key material phrase. You are ALLOWED to RE-SEGMENT the raw text to create a better (C1, T1):

Let RAW = (C1_RAW + " " + T1_RAW).

1) Produce C1 by taking the SHORTEST prefix of RAW that still:
   - explicitly mentions the key material noun/phrase (the thing whose A you will flip), AND
   - sets up the physical scene needed to interpret the action/outcome.
   You may drop irrelevant leading material from C1_RAW if it does not support the event.
   PREFERRED: C1 is ONE sentence.

2) Produce T1 as the remaining suffix that starts at the first material-dynamics event (the D you want to probe).
   You may minimally trim T1 to keep only ONE main event (remove unrelated clauses), but keep the original meaning/style.
   PREFERRED: T1 is ONE sentence.

So: you are not forced to preserve the original boundary between C1_RAW and T1_RAW.

========================
CRITICAL FIX #2: D MUST CONTRAST (NOT JUST THE MATERIAL PHRASE)
========================
T1 vs T2 must differ in a way that clearly depends on material class A.
It is NOT sufficient for T1 and T2 to differ only by swapping the material word/phrase while keeping the same dynamics.
You must change the action/outcome predicate (verb phrase or consequence) so the crossing is strong.

PREFERRED PATTERN (VERY CLEAN):
- Keep the trigger/action clause identical (e.g., "when she presses on it")
- Swap only the outcome predicate (e.g., "wrinkles" ↔ "cracks")

Examples of strong D contrasts:
- liquid: pour / drip / spill / splash / soak / douse
- granular: sprinkle / scatter / heap / pile / sift
- fabric/sheet: drape / fold / wrinkle / bunch / sag
- brittle solid: crack / shatter / chip / crumble
- soft-solid/paste: smear / spread / knead / squish / clump
Pick contrasts that make one side clearly implausible.

========================
MAIN TASK
========================
For each item:
A) Build C1 and T1 by RESEGMENTING RAW as described above.
B) Identify the key material noun/phrase in C1 (the A you will flip) and assign A1_material_class.
   IMPORTANT: the key phrase may be an entire noun phrase including the head noun (e.g., "cotton shirt"),
   not only a material adjective (so you can do "cotton shirt" ↔ "glass pane" rather than "cotton" ↔ "glass").
C) Create C2 by minimally swapping ONLY that key material noun/phrase in C1 to a different class A2,
   preserving object type plausibility (avoid impossible combos like "glass shirt" unless the phrase swap changes the head noun).
D) Keep T1 as close to the (resegmented) mined continuation as possible.
E) Create T2 by minimally editing T1 so that:
   - Under C1: T1 is clearly more plausible than T2
   - Under C2: T2 is clearly more plausible than T1
   - T2 matches T1’s skeleton: same subject/voice/tense; similar length; similar clause structure.
   - Do NOT change clause order in T2 (no moving phrases earlier/later).
   - Prefer an in-place predicate/outcome swap while keeping the trigger clause identical.
   - T1 and T2 differ primarily in the D predicate (action/outcome), not merely the key material phrase.
   - (Optional but preferred) Do not repeat the key phrase in T1/T2 if context already contains it; use pronouns like "it" where possible.

COREFERENCE RULE (IMPORTANT):
- Only use "it/they" if reference to the key object is unambiguous from C1/C2.
- If there are multiple plausible antecedents, repeat the key object phrase instead of using a pronoun.

========================
HARD CONSTRAINTS (FAIL FAST)
========================
If any of the following would be violated, output REJECT:

1) C1 and C2 must NOT be identical (after stripping whitespace/punctuation).
2) A1_material_class and A2_material_class must differ.
3) The key material noun/phrase must appear in BOTH C1 and C2 (so the A-flip is in the CONTEXT).
4) T1 and T2 must NOT be identical.
5) T1 and T2 must NOT differ only by swapping the key material phrase.
   (If the only change is oil→sand, that is a FAIL.)
6) Literal physical sense only (no idioms/metaphors; no abstract “pouring in data”).
7) Do not introduce new named entities, new tools, or new locations.
8) Do not change clause order in T2 relative to T1.

========================
FEW-SHOT EXAMPLES (FOLLOW THESE PATTERNS)
========================
Example 1 (simple EWoK-style; A in context, D in targets):
INPUT:
{"id":"EX1","C1_RAW":"AGENT-1 sees something that is fabric.","T1_RAW":"AGENT-1 can fold it."}
OUTPUT:
{"id":"EX1","C1":"AGENT-1 sees something that is fabric.","C2":"AGENT-1 sees something that is liquid.","T1":"AGENT-1 can fold it.","T2":"AGENT-1 can pour it.","A1_material_class":"fabric/sheet","A2_material_class":"liquid","key_material_noun":"fabric","notes":"A-flip fabric→liquid in context; D contrast fold→pour."}

Example 2 (resegmentation; keep purpose in context; one-sentence contexts):
INPUT:
{"id":"EX2","C1_RAW":"The head was stuffed with flax, and then steamed or subjected to the heat of a fire, while oil was","T1_RAW":"poured over it to keep it from burning."}
OUTPUT:
{"id":"EX2","C1":"The head was stuffed with flax and steamed over the heat of a fire; to keep it from burning, oil was used.","C2":"The head was stuffed with flax and steamed over the heat of a fire; to keep it from burning, cloth was used.","T1":"They pour it over the head.","T2":"They drape it over the head.","A1_material_class":"liquid","A2_material_class":"fabric/sheet","key_material_noun":"oil","notes":"Resegmented RAW so purpose stays in context. A-flip oil→cloth. D contrast pour→drape with same clause order."}

Example 3 (water basin; clean context + short D probe):
INPUT:
{"id":"EX3","C1_RAW":"Akin to both waterfalls and birdbaths, water basins are decorative bowls that catch water dripping or","T1_RAW":"pouring from a nearby spout."}
OUTPUT:
{"id":"EX3","C1":"Akin to both waterfalls and birdbaths, water basins are decorative bowls that catch water from a nearby spout.","C2":"Akin to both waterfalls and birdbaths, sand basins are decorative bowls that catch sand from a nearby spout.","T1":"Soon it drips.","T2":"Soon it piles up.","A1_material_class":"liquid","A2_material_class":"granular","key_material_noun":"water","notes":"A-flip water→sand in context. D contrast drip→pile up; same skeleton."}

Example 4 (reject metaphoric/abstract usage):
INPUT:
{"id":"EX4","C1_RAW":"New observations are pouring in from labs worldwide.","T1_RAW":"The debate is heating up."}
OUTPUT:
{"id":"EX4","REJECT":true,"reason":"Metaphorical/abstract usage of 'pouring in' (not physical material dynamics).","suggested_fix":"Mine literal physical scenes where a named substance (water/oil/sand/etc.) is poured/spilled/drips or where an object cracks/shatters/wrinkles."}

Example 5 (purpose: extrusion vs free-flow; squeeze vs pour):
INPUT:
{"id":"S1","C1_RAW":"There is thick paste in a tube.","T1_RAW":"They squeeze it out."}
OUTPUT:
{"id":"S1","C1":"There is thick paste in a tube.","C2":"There is water in a bottle.","T1":"They squeeze it out.","T2":"They pour it out.","A1_material_class":"soft-solid/paste","A2_material_class":"liquid","key_material_noun":"paste","notes":"Tests whether the model links A→D (paste-in-tube implies squeezing; liquid-in-bottle implies pouring). A-flip paste→water; D contrast squeeze out→pour out with matched skeleton and clause order."}

Example 6 (wrinkle↔crack; swap full noun phrase + swap outcome):
INPUT:
{"id":"WR1","C1_RAW":"Lena draped a cotton shirt over the back of a chair.","T1_RAW":"It wrinkles when she presses on it."}
OUTPUT:
{"id":"WR1","C1":"Lena draped a cotton shirt over the back of a chair.","C2":"Lena set a glass pane against the back of a chair.","T1":"It wrinkles when she presses on it.","T2":"It cracks when she presses on it.","A1_material_class":"fabric/sheet","A2_material_class":"brittle solid","key_material_noun":"cotton shirt","notes":"A-flip cotton shirt→glass pane in context (phrase swap includes head noun). D contrast wrinkles→cracks while keeping trigger clause identical and clause order unchanged."}

========================
OUTPUT FORMAT (JSONL ONLY)
========================
For valid items output:
{"id": <id>,
 "C1": "...", "C2": "...",
 "T1": "...", "T2": "...",
 "A1_material_class": "...", "A2_material_class": "...",
 "key_material_noun": "...",
 "notes": "A-flip + what D contrast was used (verb/outcome); mention if trigger clause was kept identical."}

For rejects:
{"id": <id>, "REJECT": true, "reason": "...", "suggested_fix": "what kind of mined example would work instead"}

Now process the following INPUT ITEMS (JSONL):"""

In [ ]:
prompt = """You are generating EWoK-style counterfactual minimal pairs for MATERIAL DYNAMICS.
(You do NOT need to know anything about EWoK—just follow the constraints below.)

INPUT: Each item has C1_RAW (mined context) and T1_RAW (mined continuation).
Your output must be (C1, C2, T1, T2) where C1/C2 contrast in material class (A) and T1/T2 contrast in dynamics or consequence (D) with strong crossed plausibility.

LATENT VIEW: z = (E, A, D, R, S)
- E: entities/objects mentioned
- A: material class/state of ONE key object/substance
- D: a physical affordance/action/outcome dependent on A
- R,S: relations/internal states; keep stable unless explicitly present

========================
ACTIVE CONCEPTS (PRIORITY)
========================
We are prioritizing these concept frames ONLY:
- stir, squeeze, hang, wrinkle

Your output MUST instantiate one of these in the literal physical scene:
- either in C1/C2 (e.g., "stirred into X", "hung on Y", "wrinkles when ...", "squeeze it ...")
- or in T1/T2 if that is the mined continuation.

If you cannot make the item clearly about one of: stir/squeeze/hang/wrinkle without heavy paraphrase, output REJECT.

========================
STRUCTURE TARGET (PREFERRED; REJECT IF NOT POSSIBLE CLEANLY)
========================
Prefer outputs where:
- C1 is exactly ONE sentence.
- C2 is exactly ONE sentence.
- T1 is exactly ONE sentence.
- T2 is exactly ONE sentence.
Do not split or merge sentences. If you cannot make this true without heavy paraphrase, output REJECT.

========================
CRITICAL FIX #1: RESEGMENT BEFORE EDITING
========================
Your mined C1_RAW may be unhelpful or not contain the key material phrase. You are ALLOWED to RE-SEGMENT the raw text to create a better (C1, T1):

Let RAW = (C1_RAW + " " + T1_RAW).

1) Produce C1 by taking the SHORTEST prefix of RAW that still:
   - explicitly mentions the key material noun/phrase (the thing whose A you will flip), AND
   - sets up the physical scene needed to interpret the action/outcome.
   You may drop irrelevant leading material from C1_RAW if it does not support the event.
   PREFERRED: C1 is ONE sentence.

2) Produce T1 as the remaining suffix that starts at the first material-dynamics event (the D you want to probe).
   You may minimally trim T1 to keep only ONE main event (remove unrelated clauses), but keep the original meaning/style.
   PREFERRED: T1 is ONE sentence.

So: you are not forced to preserve the original boundary between C1_RAW and T1_RAW.

========================
CRITICAL FIX #2: D MUST CONTRAST (NOT JUST THE MATERIAL PHRASE)
========================
T1 vs T2 must differ in a way that clearly depends on material class A.
It is NOT sufficient for T1 and T2 to differ only by swapping the material word/phrase while keeping the same dynamics.
You must change the action/outcome predicate (verb phrase or consequence) so the crossing is strong.

PREFERRED PATTERN (VERY CLEAN):
- Keep the trigger/action clause identical (e.g., "when she presses on it")
- Swap only the outcome predicate (e.g., "wrinkles" ↔ "cracks")

Examples of strong D contrasts aligned with ACTIVE CONCEPTS:
- stir-frames: blend smoothly ↔ stay clumpy / dissolve ↔ settle / mix evenly ↔ separate
- squeeze-frames: ooze out ↔ not come out / drip out ↔ stay stuck / compress ↔ crack (if brittle)
- hang-frames: droop ↔ stay rigid / sway ↔ stay still / soften in rain ↔ crack in rain
- wrinkle-frames: wrinkle/crease ↔ crack/shatter / crumple ↔ splinter

Pick contrasts that make one side clearly implausible.

========================
MAIN TASK
========================
For each item:
A) Build C1 and T1 by RESEGMENTING RAW as described above.
B) Identify the key material noun/phrase in C1 (the A you will flip) and assign A1_material_class.
   IMPORTANT: the key phrase may be an entire noun phrase including the head noun (e.g., "cotton shirt"),
   not only a material adjective (so you can do "cotton shirt" ↔ "glass pane" rather than "cotton" ↔ "glass").
C) Create C2 by minimally swapping ONLY that key material noun/phrase in C1 to a different class A2,
   preserving object type plausibility (avoid impossible combos like "glass shirt" unless the phrase swap changes the head noun).
D) Keep T1 as close to the (resegmented) mined continuation as possible.
E) Create T2 by minimally editing T1 so that:
   - Under C1: T1 is clearly more plausible than T2
   - Under C2: T2 is clearly more plausible than T1
   - T2 matches T1’s skeleton: same subject/voice/tense; similar length; similar clause structure.
   - Do NOT change clause order in T2 (no moving phrases earlier/later).
   - Prefer an in-place predicate/outcome swap while keeping the trigger clause identical.
   - T1 and T2 differ primarily in the D predicate (action/outcome), not merely the key material phrase.
   - (Optional but preferred) Do not repeat the key phrase in T1/T2 if context already contains it; use pronouns like "it" where possible.

COREFERENCE RULE (IMPORTANT):
- Only use "it/they" if reference to the key object is unambiguous from C1/C2.
- If there are multiple plausible antecedents, repeat the key object phrase instead of using a pronoun.

========================
HARD CONSTRAINTS (FAIL FAST)
========================
If any of the following would be violated, output REJECT:

1) C1 and C2 must NOT be identical (after stripping whitespace/punctuation).
2) A1_material_class and A2_material_class must differ.
3) The key material noun/phrase must appear in BOTH C1 and C2 (so the A-flip is in the CONTEXT).
4) T1 and T2 must NOT be identical.
5) T1 and T2 must NOT differ only by swapping the key material phrase.
6) Literal physical sense only (no idioms/metaphors; no abstract “pouring in data”).
7) Do not introduce new named entities, new tools, or new locations.
8) Do not change clause order in T2 relative to T1.
9) The item must clearly instantiate one ACTIVE CONCEPT (stir/squeeze/hang/wrinkle); otherwise REJECT.

========================
FEW-SHOT EXAMPLES (FOLLOW THESE PATTERNS)
========================

Example 1 (HANG; A in context, D in targets; same trigger):
INPUT:
{"id":"H1","C1_RAW":"String bars of deodorant soap on wire and hang them on the branches.","T1_RAW":"They soften in the rain."}
OUTPUT:
{"id":"H1","C1":"String bars of deodorant soap on wire and hang them on the branches.","C2":"String glass beads on wire and hang them on the branches.","T1":"They soften in the rain.","T2":"They crack in the rain.","A1_material_class":"soft-solid/paste","A2_material_class":"brittle solid","key_material_noun":"deodorant soap","notes":"ACTIVE=hang. A-flip soap bars→glass beads in same slot; D-flip soften→crack under same trigger."}

Example 2 (STIR; your clean cranberry salsa pattern):
INPUT:
{"id":"ST1","C1_RAW":"The spicy cranberry salsa can be used directly as a dip, stirred into cream cheese just before use.","T1_RAW":"It blends into a smooth mixture."}
OUTPUT:
{"id":"ST1","C1":"The spicy cranberry salsa can be used directly as a dip, stirred into cream cheese just before use.","C2":"The spicy cranberry salsa can be used directly as a dip, stirred into sand just before use.","T1":"It blends into a smooth mixture.","T2":"It stays in separate clumps.","A1_material_class":"soft-solid/paste","A2_material_class":"granular","key_material_noun":"cream cheese","notes":"ACTIVE=stir. Resegmented to keep one event; A-flip cream cheese→sand in same slot; D-flip blends smooth↔stays clumpy."}

Example 3 (SQUEEZE; avoid pour; keep concept):
INPUT:
{"id":"SQ1","C1_RAW":"There is thick paste in a tube.","T1_RAW":"They squeeze it out."}
OUTPUT:
{"id":"SQ1","C1":"There is thick paste in a tube.","C2":"There is dry sand in a jar.","T1":"They squeeze it out.","T2":"They shake it out.","A1_material_class":"soft-solid/paste","A2_material_class":"granular","key_material_noun":"paste","notes":"ACTIVE=squeeze. A-flip paste-in-tube→sand-in-jar; D-flip squeeze out↔shake out with matched skeleton."}

Example 4 (WRINKLE; swap full noun phrase + swap outcome; shared trigger clause):
INPUT:
{"id":"WR1","C1_RAW":"Lena draped a cotton shirt over the back of a chair.","T1_RAW":"It wrinkles when she presses on it."}
OUTPUT:
{"id":"WR1","C1":"Lena draped a cotton shirt over the back of a chair.","C2":"Lena set a glass pane against the back of a chair.","T1":"It wrinkles when she presses on it.","T2":"It cracks when she presses on it.","A1_material_class":"fabric/sheet","A2_material_class":"brittle solid","key_material_noun":"cotton shirt","notes":"ACTIVE=wrinkle. A-flip cotton shirt→glass pane; D-flip wrinkles→cracks while keeping trigger clause identical and clause order unchanged."}

Example 5 (REJECT metaphoric usage; use a tracked verb):
INPUT:
{"id":"RJ1","C1_RAW":"The news stirred up controversy across the country.","T1_RAW":"People argued online all night."}
OUTPUT:
{"id":"RJ1","REJECT":true,"reason":"Metaphorical/abstract usage of a tracked concept verb (stir up controversy), not physical material dynamics.","suggested_fix":"Mine literal physical scenes where stirring/squeezing/hanging/wrinkling describes physical interaction with materials."}

========================
OUTPUT FORMAT (JSONL ONLY)
========================
For valid items output:
{"id": <id>,
 "C1": "...", "C2": "...",
 "T1": "...", "T2": "...",
 "A1_material_class": "...", "A2_material_class": "...",
 "key_material_noun": "...",
 "notes": "ACTIVE=<stir|squeeze|hang|wrinkle>. A-flip + what D contrast was used."}

For rejects:
{"id": <id>, "REJECT": true, "reason": "...", "suggested_fix": "what kind of mined example would work instead"}

Now process the following INPUT ITEMS (JSONL):"""

In [ ]:
print(prompt)

In [ ]:
kept2

In [ ]:
str(kept2[3])

In [ ]:
gpt52_outputs = []
for i, item in enumerate(kept2):
    print(item)
    resp = client.responses.create(
        model="gpt-5.2",
        input=prompt + str(item),
    )
    output_text = resp.output_text
    gpt52_outputs.append(
        {
            "idx": i,
            "input_item": item,
            "output_text": output_text,
        }
    )
    print(output_text)
    print("Done")

len(gpt52_outputs)


In [ ]:
# text_only[100][0]

In [ ]:

import json
import re
import difflib
from typing import Any, List, Optional, Tuple

from transformers import AutoTokenizer

# Expanded banned patterns to match your newer prompt
BANNED_PATTERNS = [
    r"\binstead of\b",
    r"\brather than\b",
    r"\bbecause\b",
    r"\bwhich means\b",
    r"\btherefore\b",
    r"\bas a result\b",
    r"\bso that\b",
    r"\bthus\b",
    r"\bhence\b",
    r"\bin turn\b",
    r"\bmeaning\b",
    r"\bwhich makes\b",
    r"\bmaking\b",
]
banned_res = [re.compile(p, re.IGNORECASE) for p in BANNED_PATTERNS]


def extract_json_objects(text: str) -> list[dict[str, Any]]:
    decoder = json.JSONDecoder()
    objs = []
    i = 0
    while i < len(text):
        if text[i] != "{":
            i += 1
            continue
        try:
            obj, j = decoder.raw_decode(text, i)
        except json.JSONDecodeError:
            i += 1
            continue
        if isinstance(obj, dict):
            objs.append(obj)
        i = j
    return objs


def normalize_for_compare(s: str) -> str:
    return re.sub(r"[\W_]+", "", (s or "").lower())


# --- NEW: sentence counting (simple heuristic) ---
_SENT_END = re.compile(r"[.!?]")

def is_one_sentence(s: str) -> bool:
    """Heuristic: <=1 sentence-ending punctuation mark in the string."""
    if not (s and s.strip()):
        return False
    marks = _SENT_END.findall(s)
    return len(marks) <= 1


# --- NEW: word tokenization ---
def word_tokens(s: str) -> List[str]:
    # Cheap, stable. If you want punctuation-separated tokens, adjust here.
    return (s or "").strip().split()


# --- NEW: span edit check using difflib over WORD tokens ---
def span_edit_ops(a_words: List[str], b_words: List[str]) -> List[Tuple[str, int, int, int, int]]:
    sm = difflib.SequenceMatcher(a=a_words, b=b_words)
    ops = sm.get_opcodes()
    # Keep only non-equal blocks
    return [op for op in ops if op[0] != "equal"]


def passes_one_or_two_span_edit_words(
    t1: str,
    t2: str,
    one_span_max: int = 10,
    two_span_each_max: int = 6,
) -> Tuple[bool, dict]:
    """
    Pass if edits between T1 and T2 are localized to:
      - ONE contiguous edit block with both sides <= one_span_max words, OR
      - TWO edit blocks where each block has both sides <= two_span_each_max words.
    This is a good proxy for "same clause order / same skeleton".
    """
    if (t1 or "").strip() == (t2 or "").strip():
        return False, {"reason": "T1_EQ_T2"}

    a = word_tokens(t1)
    b = word_tokens(t2)
    edits = span_edit_ops(a, b)

    if not edits:
        return False, {"reason": "NO_DIFF_OPS"}

    # Number of edit blocks is the number of non-equal opcode blocks.
    k = len(edits)

    def block_lens(op):
        tag, i1, i2, j1, j2 = op
        return (i2 - i1), (j2 - j1), tag, i1, i2, j1, j2

    blocks = [block_lens(op) for op in edits]

    if k == 1:
        a_len, b_len, tag, i1, i2, j1, j2 = blocks[0]
        ok = (a_len <= one_span_max and b_len <= one_span_max)
        meta = {
            "mode": "one_span",
            "a_span_len_words": a_len,
            "b_span_len_words": b_len,
            "opcode": (tag, i1, i2, j1, j2),
            "a_span_text": " ".join(a[i1:i2]),
            "b_span_text": " ".join(b[j1:j2]),
        }
        return ok, (meta if ok else {**meta, "reason": "SPAN_TOO_LONG"})

    if k == 2:
        ok_each = True
        metas = []
        for (a_len, b_len, tag, i1, i2, j1, j2) in blocks:
            ok_each = ok_each and (a_len <= two_span_each_max and b_len <= two_span_each_max)
            metas.append({
                "a_span_len_words": a_len,
                "b_span_len_words": b_len,
                "opcode": (tag, i1, i2, j1, j2),
                "a_span_text": " ".join(a[i1:i2]),
                "b_span_text": " ".join(b[j1:j2]),
            })
        meta = {"mode": "two_span", "blocks": metas}
        return ok_each, (meta if ok_each else {**meta, "reason": "SPAN_TOO_LONG"})

    return False, {"reason": "TOO_MANY_EDIT_BLOCKS", "num_blocks": k}


# --- NEW: infer swapped phrase in WORD space (aligns with "swap noun phrase") ---
def infer_swap_phrase_from_context_words(c1: str, c2: str, max_span_words: int = 12) -> Optional[str]:
    a = word_tokens(c1)
    b = word_tokens(c2)
    edits = span_edit_ops(a, b)
    if len(edits) != 1:
        return None
    tag, i1, i2, j1, j2 = edits[0]
    a_span = a[i1:i2]
    b_span = b[j1:j2]
    if len(a_span) == 0 and len(b_span) == 0:
        return None
    if len(a_span) > max_span_words or len(b_span) > max_span_words:
        return None
    phrase = " ".join(b_span).strip()
    return phrase if phrase else None


def build_verifier_detail(result: dict[str, Any]) -> str:
    if result.get("status") == "model_reject":
        return (
            f"idx={result.get('idx')} pass=False status=model_reject "
            f"reason={result.get('reason', '')}"
        )
    if result.get("status") == "parse_error":
        return (
            f"idx={result.get('idx')} pass=False status=parse_error "
            f"reason={result.get('reason', '')}"
        )
    return (
        f"idx={result.get('idx')} pass={result.get('all_checks_pass')} "
        f"status={result.get('status')} "
        f"sent_ok={result.get('one_sentence_ok')} "
        f"ctx_ok={result.get('context_checks_ok')} "
        f"c1!=c2={result.get('c1_ne_c2_norm')} "
        f"a_ok={result.get('a_classes_differ')} "
        f"key_in_c1={result.get('key_in_c1')} "
        f"swap_in_c2={result.get('swap_in_c2')} "
        f"swap!=key={result.get('swap_differs_from_key')} "
        f"edit_ok={result.get('edit_ok')} "
        f"len_ok={result.get('t2_length_ok')} "
        f"only_mat_swap_fail={result.get('t1t2_only_material_swap_fail')} "
        f"banned_ok={result.get('no_banned_patterns_in_t2')}"
    )


def run_verifier(gpt_outputs: list[dict[str, Any]], tokenizer_name: str = "gpt2") -> dict[str, Any]:
    # Still keep tokenizer around if you want, but core checks now use word space.
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

    check_results = []
    for rec in gpt_outputs:
        objs = extract_json_objects(rec.get("output_text", ""))
        if not objs:
            row = {
                "idx": rec.get("idx"),
                "status": "parse_error",
                "all_checks_pass": False,
                "reason": "Could not parse JSON object from model output.",
                "input_item": rec.get("input_item"),
            }
            row["verifier_detail"] = build_verifier_detail(row)
            check_results.append(row)
            continue

        obj = objs[0]
        if obj.get("REJECT") is True:
            row = {
                "idx": rec.get("idx"),
                "status": "model_reject",
                "all_checks_pass": False,
                "reason": obj.get("reason", ""),
                "suggested_fix": obj.get("suggested_fix", ""),
                "input_item": rec.get("input_item"),
                "parsed_output": obj,
            }
            row["verifier_detail"] = build_verifier_detail(row)
            check_results.append(row)
            continue

        c1 = obj.get("C1", "")
        c2 = obj.get("C2", "")
        t1 = obj.get("T1", "")
        t2 = obj.get("T2", "")
        key = obj.get("key_material_noun", "")
        a1 = obj.get("A1_material_class", "")
        a2 = obj.get("A2_material_class", "")

        # One-sentence constraint (heuristic)
        one_sentence_ok = all(is_one_sentence(x) for x in [c1, c2, t1, t2])

        c1_ne_c2 = normalize_for_compare(c1) != normalize_for_compare(c2)

        key_norm = normalize_for_compare(key)
        key_in_c1 = bool(key_norm) and key_norm in normalize_for_compare(c1)

        # Infer swap phrase in WORD space (better for noun-phrase swaps)
        swap_from_output = (obj.get("swap_material_noun") or "").strip()
        if swap_from_output:
            swap_phrase = swap_from_output
            swap_source = "output"
        else:
            swap_phrase = infer_swap_phrase_from_context_words(c1, c2)
            swap_source = "inferred_words" if swap_phrase else "none"

        swap_norm = normalize_for_compare(swap_phrase or "")
        swap_in_c2 = bool(swap_norm) and swap_norm in normalize_for_compare(c2)
        swap_differs_from_key = bool(swap_norm) and swap_norm != key_norm

        # A-class check
        a_classes_differ = bool(a1.strip()) and bool(a2.strip()) and (a1.strip() != a2.strip())

        # Length similarity in WORDS
        t1_words = len(word_tokens(t1))
        t2_words = len(word_tokens(t2))
        t2_len_ok = abs(t2_words - t1_words) <= 6  # simple "similar length" proxy

        # Edit locality proxy (also enforces "no clause order change" fairly well)
        edit_ok, edit_meta = passes_one_or_two_span_edit_words(t1, t2, one_span_max=10, two_span_each_max=6)

        # “Not only material swap” check (only meaningful if key/swap appears in T1/T2)
        # If the only edit block is swapping key<->swap, fail.
        t1_norm = normalize_for_compare(t1)
        t2_norm = normalize_for_compare(t2)
        t1t2_only_material_swap_fail = False
        if edit_ok:
            # If key/swap appear in targets, detect trivial swaps
            if (key_norm and key_norm in t1_norm) or (swap_norm and swap_norm in t1_norm) or \
               (key_norm and key_norm in t2_norm) or (swap_norm and swap_norm in t2_norm):
                # In one-span mode, check if span texts are basically key/swap
                if edit_meta.get("mode") == "one_span":
                    a_span_n = normalize_for_compare(edit_meta.get("a_span_text", ""))
                    b_span_n = normalize_for_compare(edit_meta.get("b_span_text", ""))
                    trivial = (
                        (a_span_n == key_norm and b_span_n == swap_norm) or
                        (a_span_n == swap_norm and b_span_n == key_norm)
                    )
                    t1t2_only_material_swap_fail = trivial

        banned_hits = [p.pattern for p in banned_res if p.search(t2)]
        no_banned_patterns = len(banned_hits) == 0

        context_checks_ok = c1_ne_c2 and key_in_c1 and swap_in_c2 and swap_differs_from_key and a_classes_differ
        all_checks_pass = (
            one_sentence_ok and
            context_checks_ok and
            t2_len_ok and
            edit_ok and
            no_banned_patterns and
            (not t1t2_only_material_swap_fail)
        )

        row = {
            "idx": rec.get("idx"),
            "status": "ok",
            "all_checks_pass": all_checks_pass,

            "one_sentence_ok": one_sentence_ok,

            "context_checks_ok": context_checks_ok,
            "c1_ne_c2_norm": c1_ne_c2,
            "a_classes_differ": a_classes_differ,
            "key_in_c1": key_in_c1,

            "swap_phrase": swap_phrase,
            "swap_source": swap_source,
            "swap_in_c2": swap_in_c2,
            "swap_differs_from_key": swap_differs_from_key,

            "edit_ok": edit_ok,
            "edit_meta": edit_meta,

            "t1_words": t1_words,
            "t2_words": t2_words,
            "t2_length_ok": t2_len_ok,

            "t1t2_only_material_swap_fail": t1t2_only_material_swap_fail,

            "no_banned_patterns_in_t2": no_banned_patterns,
            "banned_pattern_hits": banned_hits,

            "C1": c1,
            "C2": c2,
            "T1": t1,
            "T2": t2,
            "key_material_noun": key,
            "A1_material_class": a1,
            "A2_material_class": a2,

            "input_item": rec.get("input_item"),
            "parsed_output": obj,
        }
        row["verifier_detail"] = build_verifier_detail(row)
        check_results.append(row)

    counts = {
        "total": len(check_results),
        "pass_true": sum(1 for r in check_results if r.get("all_checks_pass") is True),
        "pass_false": sum(1 for r in check_results if r.get("all_checks_pass") is False),
        "status_ok": sum(1 for r in check_results if r.get("status") == "ok"),
        "status_model_reject": sum(1 for r in check_results if r.get("status") == "model_reject"),
        "status_parse_error": sum(1 for r in check_results if r.get("status") == "parse_error"),
    }

    return {
        "tokenizer_name": tokenizer_name,
        "check_results": check_results,
        "counts": counts,
    }

In [ ]:
from copy import deepcopy


def partition_verifier_groups(verification_payload: dict[str, Any]) -> dict[str, Any]:
    check_results = verification_payload.get("check_results", [])

    passed_verifier = [
        r for r in check_results if r.get("all_checks_pass") is True
    ]

    failed_model_reject = [
        r
        for r in check_results
        if r.get("all_checks_pass") is False and r.get("status") == "model_reject"
    ]

    failed_status_ok = [
        r
        for r in check_results
        if r.get("all_checks_pass") is False and r.get("status") == "ok"
    ]

    other_failures = [
        r
        for r in check_results
        if r.get("all_checks_pass") is False
        and r.get("status") not in {"ok", "model_reject"}
    ]

    grouped = {
        "passed_verifier": passed_verifier,
        "failed_model_reject": failed_model_reject,
        "failed_status_ok": failed_status_ok,
        "other_failures": other_failures,
        "counts": {
            "passed_verifier": len(passed_verifier),
            "failed_model_reject": len(failed_model_reject),
            "failed_status_ok": len(failed_status_ok),
            "other_failures": len(other_failures),
            "total": len(check_results),
        },
    }
    return grouped


def compact_check_results_rows(
    rows: list[dict[str, Any]],
    keep_input_item: bool = False,
    keep_parsed_output: bool = False,
) -> list[dict[str, Any]]:
    """
    Remove redundant top-level fields (C1/C2/T1/T2 duplicates, etc.) while
    preserving verifier diagnostics and one canonical pair payload.
    """
    compact_rows = []
    for row in rows:
        parsed = row.get("parsed_output") or {}

        pair = {
            "id": parsed.get("id"),
            "C1": parsed.get("C1", row.get("C1")),
            "C2": parsed.get("C2", row.get("C2")),
            "T1": parsed.get("T1", row.get("T1")),
            "T2": parsed.get("T2", row.get("T2")),
            "key_material_noun": parsed.get("key_material_noun", row.get("key_material_noun")),
            "A1_material_class": parsed.get("A1_material_class", row.get("A1_material_class")),
            "A2_material_class": parsed.get("A2_material_class", row.get("A2_material_class")),
            "notes": parsed.get("notes"),
        }

        checks = {
            "one_sentence_ok": row.get("one_sentence_ok"),
            "context_checks_ok": row.get("context_checks_ok"),
            "c1_ne_c2_norm": row.get("c1_ne_c2_norm"),
            "a_classes_differ": row.get("a_classes_differ"),
            "key_in_c1": row.get("key_in_c1"),
            "swap_phrase": row.get("swap_phrase"),
            "swap_source": row.get("swap_source"),
            "swap_in_c2": row.get("swap_in_c2"),
            "swap_differs_from_key": row.get("swap_differs_from_key"),
            # Backward/forward compatibility for old/new verifier field names:
            "single_span_edit_ok": row.get("single_span_edit_ok", row.get("edit_ok")),
            "single_span_meta": row.get("single_span_meta", row.get("edit_meta")),
            "t1_tokens": row.get("t1_tokens", row.get("t1_words")),
            "t2_tokens": row.get("t2_tokens", row.get("t2_words")),
            "t2_length_ok": row.get("t2_length_ok"),
            "t1t2_only_material_swap_fail": row.get("t1t2_only_material_swap_fail"),
            "no_banned_patterns_in_t2": row.get("no_banned_patterns_in_t2"),
            "banned_pattern_hits": row.get("banned_pattern_hits"),
        }

        compact = {
            "idx": row.get("idx"),
            "status": row.get("status"),
            "all_checks_pass": row.get("all_checks_pass"),
            "reason": row.get("reason", ""),
            "suggested_fix": row.get("suggested_fix", ""),
            "verifier_detail": row.get("verifier_detail", ""),
            "pair": pair,
            "checks": checks,
        }

        if keep_input_item:
            compact["input_item"] = deepcopy(row.get("input_item"))
        if keep_parsed_output:
            compact["parsed_output"] = deepcopy(row.get("parsed_output"))

        compact_rows.append(compact)

    return compact_rows


def compact_partitioned_results(
    partitioned: dict[str, Any],
    keep_input_item: bool = False,
    keep_parsed_output: bool = False,
) -> dict[str, Any]:
    out = {
        "counts": dict(partitioned.get("counts", {})),
        "passed_verifier": compact_check_results_rows(
            partitioned.get("passed_verifier", []),
            keep_input_item=keep_input_item,
            keep_parsed_output=keep_parsed_output,
        ),
        "failed_model_reject": compact_check_results_rows(
            partitioned.get("failed_model_reject", []),
            keep_input_item=keep_input_item,
            keep_parsed_output=keep_parsed_output,
        ),
        "failed_status_ok": compact_check_results_rows(
            partitioned.get("failed_status_ok", []),
            keep_input_item=keep_input_item,
            keep_parsed_output=keep_parsed_output,
        ),
        "other_failures": compact_check_results_rows(
            partitioned.get("other_failures", []),
            keep_input_item=keep_input_item,
            keep_parsed_output=keep_parsed_output,
        ),
    }
    return out


# This is the missing step in your current notebook state:
# compute check_results from gpt52_outputs before compacting.
verification_payload = run_verifier(gpt52_outputs)
check_results = verification_payload["check_results"]
partitioned_results = partition_verifier_groups(verification_payload)

compact_check_results = compact_check_results_rows(check_results)
compact_partitioned_results_payload = compact_partitioned_results(partitioned_results)
compact_partitioned_results_payload["counts"]


In [ ]:
compact_partitioned_results_payload["passed_verifier"]

In [ ]:
compact_partitioned_results_payload["failed_model_reject"]

In [ ]:
partitioned_results['failed_status_ok']

In [ ]:
editor_prompt =  """You are EDITOR-2 for counterfactual minimal pairs about PHYSICAL / MATERIAL BEHAVIOR (MATERIAL DYNAMICS).

ROLE:
Repair failed examples. Do NOT generate from scratch.
Preserve mined integrity: keep C1/C2 and T1 as close as possible (ideally identical),
and only make small, localized edits (mostly to T2).

HARD STRUCTURE CONSTRAINT (MUST HOLD):
- C1 is exactly ONE sentence.
- C2 is exactly ONE sentence.
- T1 is exactly ONE sentence.
- T2 is exactly ONE sentence.
Do not split or merge sentences.

IMPORTANT DEFINITIONS:
- "token" = one whitespace-delimited word.
- "outside edited spans, identical" means character-for-character identical (same punctuation/case).

WORKFLOW (follow exactly):
1) Copy T1 exactly (character-for-character).
2) Create T2 by applying ONLY the allowed span replacement(s) to that copied T1.
3) Do not change anything else in T2.

LATENT STATE (self-contained; use for reasoning, do not print):
Each context C induces a latent state z = (E, A, R, D, I):

- E (Entities): the concrete objects/agents mentioned (e.g., water, sand, pipe, drain, fire, bottle).
- A (Attributes / Material class): the material category or physical state of the key substance/object
  (e.g., liquid, granular, fabric/sheet, brittle solid, soft-solid/paste).
- R (Relations): physical relations between entities (in/into/on/over/under/inside, containment, location).
  Example: “water in the bottle”, “sand in the drain”, “poured over it”.
- D (Dynamics / Updates): the action or outcome that should depend on A (pour, spill, soak, smother, settle, pile).
  This is what T1 vs T2 should probe.
- I (Intent / Instruction context): goals or procedural framing if present (e.g., “to extinguish the fire”).
  I should remain unchanged; do NOT add new goals or explanations.

GOAL:
You are given (C1, C2, T1, T2). C1 and C2 should differ mainly in A (material class) while keeping E and R stable.
T1 and T2 should differ mainly in D (a material-dependent action/outcome), so plausibility flips across contexts:
- Under C1: T1 should be clearly more plausible than T2.
- Under C2: T2 should be clearly more plausible than T1.

INPUT:
Each item is a JSON object containing at least:
{id, C1, C2, T1, T2, key_material_noun, A1_material_class, A2_material_class, notes}
It may also include verifier failures such as SPAN_TOO_LONG or banned pattern hits.

OUTPUT (JSONL only):
For each input item, output exactly ONE JSON object on its own line.

If repaired:
{"id":..., "C1":"...","C2":"...","T1":"...","T2":"...","A1_material_class":"...","A2_material_class":"...","key_material_noun":"...","notes":"REPAIR: ..."}
If not repairable under Track-1.5:
{"id":..., "REJECT":true,"reason":"...","suggested_fix":"..."}

IMPORTANT COPY RULES:
- Copy `A1_material_class`, `A2_material_class`, and `key_material_noun` exactly from the input; do not change them.
- Preserve C1/C2 exactly unless the A-flip is broken. Prefer leaving C1/C2 unchanged.
- Preserve T1 exactly unless T1 contains forbidden discourse markers or is inconsistent with C2.

========================
TRACK-1.5 RULES
========================
1) Literal physical sense only (no metaphors/idioms).
2) Keep E and R stable: do NOT add new named entities, tools, locations, or extra events.
3) Preserve C1 and C2 unless the A-flip is broken. Prefer leaving C1/C2 unchanged.
4) Preserve T1 exactly unless T1 itself contains banned discourse markers or is inconsistent with C2.
5) Edit budget for T2 relative to T1 (token = whitespace-delimited word):
   - T2 must be obtainable from T1 by either:
     (a) ONE contiguous span replacement of <= 10 tokens, OR
     (b) TWO contiguous span replacements, each <= 6 tokens.
   Insertions/deletions are allowed only inside those span(s).
   Outside the edited span(s), keep T2 identical to T1 (character-for-character).
6) Do not change clause order or sentence skeleton in T2:
   - Same clause order as T1.
   - Same tense/voice as T1.
   - Do not move phrases earlier/later; only replace within a span in-place.
7) No explanatory discourse markers in T2 (no commentary/explanation):
   Forbidden: "instead of", "rather than", "because", "which means", "therefore", "so that", "as a result".
   Also avoid: "thus", "hence", "making", "meaning", "in turn", "which makes".
8) No new facts:
   - Avoid adding new mechanism nouns (e.g., "oxygen", "air", "pressure") unless they already appear in T1.
   - Prefer verb/adjective swaps (begin↔stop, soak↔smother, sink↔float, fizz↔clump).

If you cannot satisfy these constraints without heavy paraphrase or adding new facts/entities, output REJECT.

========================
HOW TO FIX COMMON FAILURES
========================
- If failure is SPAN_TOO_LONG: compress T2 into a closer mirror of T1, using at most 1–2 short predicate swaps.
- If T2 introduces mechanism nouns (e.g., "air"): remove them; prefer a verb swap within the same sentence skeleton.
- If C2 swap collapses the contrast (e.g., oil with petroleum): REJECT unless a minimal clean swap is possible.

========================
FEW-SHOT EXAMPLES (FOLLOW THESE REPAIR STYLES)
========================

Example E2-1 (two-span edit accepted; keep chunks together; no extra facts)
INPUT:
{"id":14,
 "C1":"If a pipe breaks, gravity will carry the water",
 "C2":"If a pipe breaks, gravity will carry the sand",
 "T1":"pouring from its broken ends down to the containment sump.",
 "T2":"spilling from its broken ends and piling up in the containment sump.",
 "A1_material_class":"liquid","A2_material_class":"granular","key_material_noun":"water",
 "notes":"A-flip water→sand in context; D contrast flow/pour vs spill/pile."}
OUTPUT:
{"id":14,
 "C1":"If a pipe breaks, gravity will carry the water",
 "C2":"If a pipe breaks, gravity will carry the sand",
 "T1":"pouring from its broken ends down to the containment sump.",
 "T2":"spilling from its broken ends and piling up in the containment sump.",
 "A1_material_class":"liquid","A2_material_class":"granular","key_material_noun":"water",
 "notes":"REPAIR: kept as-is; already a clean A→D contrast. Two-span edit vs T1: pouring→spilling; down to→and piling up in."
}

Example E2-2 (compress T2 to match T1 skeleton; minimal negation; no new facts)
INPUT:
{"id":"unknown",
 "C1":"When you’ve attached the balloon well, start pouring the soda into the bottle, where it will mix with the vinegar.",
 "C2":"When you’ve attached the balloon well, start pouring the soda into the bottle, where it will mix with the sand.",
 "T1":"A chemical reaction will begin.",
 "T2":"It will just settle to the bottom.",
 "A1_material_class":"liquid","A2_material_class":"granular","key_material_noun":"vinegar",
 "notes":"A-flip vinegar→sand; want D contrast without extra facts."}
OUTPUT:
{"id":"unknown",
 "C1":"When you’ve attached the balloon well, start pouring the soda into the bottle, where it will mix with the vinegar.",
 "C2":"When you’ve attached the balloon well, start pouring the soda into the bottle, where it will mix with the sand.",
 "T1":"A chemical reaction will begin.",
 "T2":"A reaction will not begin.",
 "A1_material_class":"liquid","A2_material_class":"granular","key_material_noun":"vinegar",
 "notes":"REPAIR: edited only T2 with a local negation (begin→not begin) to enforce crossed plausibility; no added mechanism text."
}

Example E2-3 (remove mechanism noun; keep T1; single predicate swap)
INPUT:
{"id":"RULE3_EXTINGUISH",
 "C1":"Properly and fully extinguish the fire: pouring water on a fire in a slow and controlled fashion is the ideal method.",
 "C2":"Properly and fully extinguish the fire: pouring sand on a fire in a slow and controlled fashion is the ideal method.",
 "T1":"It soaks the flames and cools them down.",
 "T2":"It smothers the flames and cuts off the air.",
 "A1_material_class":"liquid","A2_material_class":"granular","key_material_noun":"water",
 "notes":"Avoid adding mechanism nouns like 'air'."}
OUTPUT:
{"id":"RULE3_EXTINGUISH",
 "C1":"Properly and fully extinguish the fire: pouring water on a fire in a slow and controlled fashion is the ideal method.",
 "C2":"Properly and fully extinguish the fire: pouring sand on a fire in a slow and controlled fashion is the ideal method.",
 "T1":"It soaks the flames and cools them down.",
 "T2":"It smothers the flames and cools them down.",
 "A1_material_class":"liquid","A2_material_class":"granular","key_material_noun":"water",
 "notes":"REPAIR: edited only T2; local predicate swap (soaks→smothers). Removed extra mechanism phrase."
}

========================
NOW REPAIR THE FOLLOWING ITEMS (JSONL)
======================== """

In [ ]:
# import json


# def _extract_json_objects_from_text(text: str) -> list[dict]:
#     decoder = json.JSONDecoder()
#     objs = []
#     i = 0
#     while i < len(text):
#         if text[i] != "{":
#             i += 1
#             continue
#         try:
#             obj, j = decoder.raw_decode(text, i)
#         except json.JSONDecodeError:
#             i += 1
#             continue
#         if isinstance(obj, dict):
#             objs.append(obj)
#         i = j
#     return objs


# def _build_editor_input(compact_row: dict) -> dict:
#     pair = compact_row.get("pair", {})
#     return {
#         "id": pair.get("id", compact_row.get("idx")),
#         "C1": pair.get("C1", ""),
#         "C2": pair.get("C2", ""),
#         "T1": pair.get("T1", ""),
#         "T2": pair.get("T2", ""),
#         "A1_material_class": pair.get("A1_material_class", ""),
#         "A2_material_class": pair.get("A2_material_class", ""),
#         "key_material_noun": pair.get("key_material_noun", ""),
#         "notes": pair.get("notes") or compact_row.get("verifier_detail", ""),
#         "verifier": compact_row.get("checks", {}),
#         "verifier_detail": compact_row.get("verifier_detail", ""),
#     }


# failed_status_ok_items = compact_partitioned_results_payload["failed_status_ok"]

# editor_gpt52_outputs = []
# for row in failed_status_ok_items:
#     editor_input_item = _build_editor_input(row)
#     resp = client.responses.create(
#         model="gpt-5.2",
#         input=f"{editor_prompt}\n{json.dumps(editor_input_item, ensure_ascii=False)}",
#     )
#     output_text = resp.output_text
#     editor_gpt52_outputs.append(
#         {
#             "idx": row.get("idx"),
#             "input_item": editor_input_item,
#             "output_text": output_text,
#         }
#     )

# editor_gpt52_results = []
# for rec in editor_gpt52_outputs:
#     parsed = _extract_json_objects_from_text(rec["output_text"])
#     if not parsed:
#         editor_gpt52_results.append(
#             {
#                 "idx": rec["idx"],
#                 "status": "parse_error",
#                 "input_item": rec["input_item"],
#                 "output_text": rec["output_text"],
#             }
#         )
#         continue

#     obj = parsed[0]
#     if obj.get("REJECT") is True:
#         editor_gpt52_results.append(
#             {
#                 "idx": rec["idx"],
#                 "status": "rejected",
#                 "input_item": rec["input_item"],
#                 "rejected": obj,
#                 "output_text": rec["output_text"],
#             }
#         )
#     else:
#         editor_gpt52_results.append(
#             {
#                 "idx": rec["idx"],
#                 "status": "repaired",
#                 "input_item": rec["input_item"],
#                 "repaired": obj,
#                 "output_text": rec["output_text"],
#             }
#         )

# editor_gpt52_repaired = [r for r in editor_gpt52_results if r["status"] == "repaired"]
# editor_gpt52_rejected = [r for r in editor_gpt52_results if r["status"] == "rejected"]
# editor_gpt52_parse_errors = [r for r in editor_gpt52_results if r["status"] == "parse_error"]

# editor_gpt52_payload = {
#     "counts": {
#         "total_inputs": len(failed_status_ok_items),
#         "repaired": len(editor_gpt52_repaired),
#         "rejected": len(editor_gpt52_rejected),
#         "parse_error": len(editor_gpt52_parse_errors),
#     },
#     "repaired": editor_gpt52_repaired,
#     "rejected": editor_gpt52_rejected,
#     "parse_error": editor_gpt52_parse_errors,
#     "all_results": editor_gpt52_results,
# }

# editor_gpt52_payload["counts"]
import json


def _extract_json_objects_from_text(text: str) -> list[dict]:
    decoder = json.JSONDecoder()
    objs = []
    i = 0
    while i < len(text):
        if text[i] != "{":
            i += 1
            continue
        try:
            obj, j = decoder.raw_decode(text, i)
        except json.JSONDecodeError:
            i += 1
            continue
        if isinstance(obj, dict):
            objs.append(obj)
        i = j
    return objs


def _build_editor_input(compact_row: dict) -> dict:
    # Backward-compatible: your compact rows might store the model pair in different fields
    pair = (
        compact_row.get("pair")
        or compact_row.get("parsed_output")
        or compact_row.get("repaired")
        or {}
    )

    # Backward-compatible: old pipeline used `checks`; new verifier likely stores flags at top-level
    verifier_checks = compact_row.get("checks")
    if verifier_checks is None:
        # Keep this minimal: just forward the most useful top-level verifier fields if present
        verifier_checks = {
            k: compact_row.get(k)
            for k in [
                "one_sentence_ok",
                "context_checks_ok",
                "c1_ne_c2_norm",
                "a_classes_differ",
                "key_in_c1",
                "swap_phrase",
                "swap_source",
                "swap_in_c2",
                "swap_differs_from_key",
                "edit_ok",
                "edit_meta",
                "t1_words",
                "t2_words",
                "t2_length_ok",
                "t1t2_only_material_swap_fail",
                "no_banned_patterns_in_t2",
                "banned_pattern_hits",
            ]
            if k in compact_row
        }

    return {
        "id": pair.get("id", compact_row.get("idx")),
        "C1": pair.get("C1", compact_row.get("C1", "")),
        "C2": pair.get("C2", compact_row.get("C2", "")),
        "T1": pair.get("T1", compact_row.get("T1", "")),
        "T2": pair.get("T2", compact_row.get("T2", "")),
        "A1_material_class": pair.get("A1_material_class", compact_row.get("A1_material_class", "")),
        "A2_material_class": pair.get("A2_material_class", compact_row.get("A2_material_class", "")),
        "key_material_noun": pair.get("key_material_noun", compact_row.get("key_material_noun", "")),
        "notes": pair.get("notes") or compact_row.get("verifier_detail", ""),
        "verifier": verifier_checks,
        "verifier_detail": compact_row.get("verifier_detail", ""),
    }


failed_status_ok_items = compact_partitioned_results_payload["failed_status_ok"]

editor_gpt52_outputs = []
for row in failed_status_ok_items:
    editor_input_item = _build_editor_input(row)
    resp = client.responses.create(
        model="gpt-5.2",
        input=f"{editor_prompt}\n{json.dumps(editor_input_item, ensure_ascii=False)}",
    )
    output_text = resp.output_text
    editor_gpt52_outputs.append(
        {
            "idx": row.get("idx"),
            "input_item": editor_input_item,
            "output_text": output_text,
        }
    )

editor_gpt52_results = []
for rec in editor_gpt52_outputs:
    parsed = _extract_json_objects_from_text(rec["output_text"])
    if not parsed:
        editor_gpt52_results.append(
            {
                "idx": rec["idx"],
                "status": "parse_error",
                "input_item": rec["input_item"],
                "output_text": rec["output_text"],
            }
        )
        continue

    obj = parsed[0]
    if obj.get("REJECT") is True:
        editor_gpt52_results.append(
            {
                "idx": rec["idx"],
                "status": "rejected",
                "input_item": rec["input_item"],
                "rejected": obj,
                "output_text": rec["output_text"],
            }
        )
    else:
        editor_gpt52_results.append(
            {
                "idx": rec["idx"],
                "status": "repaired",
                "input_item": rec["input_item"],
                "repaired": obj,
                "output_text": rec["output_text"],
            }
        )

editor_gpt52_repaired = [r for r in editor_gpt52_results if r["status"] == "repaired"]
editor_gpt52_rejected = [r for r in editor_gpt52_results if r["status"] == "rejected"]
editor_gpt52_parse_errors = [r for r in editor_gpt52_results if r["status"] == "parse_error"]

editor_gpt52_payload = {
    "counts": {
        "total_inputs": len(failed_status_ok_items),
        "repaired": len(editor_gpt52_repaired),
        "rejected": len(editor_gpt52_rejected),
        "parse_error": len(editor_gpt52_parse_errors),
    },
    "repaired": editor_gpt52_repaired,
    "rejected": editor_gpt52_rejected,
    "parse_error": editor_gpt52_parse_errors,
    "all_results": editor_gpt52_results,
}

editor_gpt52_payload["counts"]

In [ ]:
editor_gpt52_outputs[3]['output_text']

In [ ]:
# import json
# from datetime import datetime, timezone
# from pathlib import Path


# def _pair_from_compact_row(row: dict, source_group: str) -> dict:
#     pair = row.get("pair", {})
#     return {
#         "id": pair.get("id", row.get("idx")),
#         "idx": row.get("idx"),
#         "source_group": source_group,
#         "C1": pair.get("C1", ""),
#         "C2": pair.get("C2", ""),
#         "T1": pair.get("T1", ""),
#         "T2": pair.get("T2", ""),
#         "A1_material_class": pair.get("A1_material_class", ""),
#         "A2_material_class": pair.get("A2_material_class", ""),
#         "key_material_noun": pair.get("key_material_noun", ""),
#         "notes": pair.get("notes", ""),
#     }


# def _pair_from_repaired_row(row: dict, source_group: str = "repaired") -> dict:
#     repaired = row.get("repaired", {}) or {}
#     input_item = row.get("input_item", {}) or {}
#     return {
#         "id": repaired.get("id", input_item.get("id", row.get("idx"))),
#         "idx": row.get("idx"),
#         "source_group": source_group,
#         "C1": repaired.get("C1", input_item.get("C1", "")),
#         "C2": repaired.get("C2", input_item.get("C2", "")),
#         "T1": repaired.get("T1", input_item.get("T1", "")),
#         "T2": repaired.get("T2", input_item.get("T2", "")),
#         "A1_material_class": repaired.get("A1_material_class", input_item.get("A1_material_class", "")),
#         "A2_material_class": repaired.get("A2_material_class", input_item.get("A2_material_class", "")),
#         "key_material_noun": repaired.get("key_material_noun", input_item.get("key_material_noun", "")),
#         "notes": repaired.get("notes", input_item.get("notes", "")),
#     }


# def _dedupe_pairs(pairs: list[dict]) -> list[dict]:
#     seen = set()
#     out = []
#     for p in pairs:
#         key = (
#             p.get("id"),
#             p.get("C1"),
#             p.get("C2"),
#             p.get("T1"),
#             p.get("T2"),
#         )
#         if key in seen:
#             continue
#         seen.add(key)
#         out.append(p)
#     return out


# def _write_jsonl(path: Path, rows: list[dict]) -> None:
#     with path.open("w", encoding="utf-8") as f:
#         for r in rows:
#             f.write(json.dumps(r, ensure_ascii=False) + "\n")


# if "compact_partitioned_results_payload" not in globals():
#     raise NameError("compact_partitioned_results_payload not found. Run the compacting cell first.")
# if "editor_gpt52_payload" not in globals():
#     raise NameError("editor_gpt52_payload not found. Run the editor repair cell first.")

# passed_rows = compact_partitioned_results_payload.get("passed_verifier", [])
# repaired_rows = editor_gpt52_payload.get("repaired", [])

# passed_pairs_for_aug = [_pair_from_compact_row(r, "passed_verifier") for r in passed_rows]
# repaired_pairs_for_aug = [_pair_from_repaired_row(r, "repaired") for r in repaired_rows]
# merged_pairs_for_aug = _dedupe_pairs(passed_pairs_for_aug + repaired_pairs_for_aug)

# out_dir = Path("data_augmentation")
# out_dir.mkdir(parents=True, exist_ok=True)

# passed_jsonl = out_dir / "context_target_pairs_passed.jsonl"
# repaired_jsonl = out_dir / "context_target_pairs_repaired.jsonl"
# merged_jsonl = out_dir / "context_target_pairs_merged.jsonl"
# merged_json = out_dir / "context_target_pairs_merged.json"
# manifest_json = out_dir / "manifest.json"

# _write_jsonl(passed_jsonl, passed_pairs_for_aug)
# _write_jsonl(repaired_jsonl, repaired_pairs_for_aug)
# _write_jsonl(merged_jsonl, merged_pairs_for_aug)
# merged_json.write_text(json.dumps(merged_pairs_for_aug, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

# manifest = {
#     "created_at_utc": datetime.now(timezone.utc).isoformat(),
#     "output_dir": str(out_dir),
#     "counts": {
#         "passed_pairs": len(passed_pairs_for_aug),
#         "repaired_pairs": len(repaired_pairs_for_aug),
#         "merged_pairs": len(merged_pairs_for_aug),
#     },
#     "files": {
#         "passed_jsonl": str(passed_jsonl),
#         "repaired_jsonl": str(repaired_jsonl),
#         "merged_jsonl": str(merged_jsonl),
#         "merged_json": str(merged_json),
#     },
# }
# manifest_json.write_text(json.dumps(manifest, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

# manifest
import json
from datetime import datetime, timezone
from pathlib import Path


def _pair_from_compact_row(row: dict, source_group: str) -> dict:
    # Backward compatible: pair might live in different places depending on your pipeline stage
    pair = (
        row.get("pair")
        or row.get("parsed_output")
        or row.get("repaired")
        or {}
    )

    # Optional: carry minimal verifier info if present (new verifier puts flags at top-level)
    verifier = row.get("checks")
    if verifier is None:
        verifier = {
            k: row.get(k)
            for k in [
                "one_sentence_ok",
                "context_checks_ok",
                "a_classes_differ",
                "edit_ok",
                "t2_length_ok",
                "no_banned_patterns_in_t2",
                "banned_pattern_hits",
            ]
            if k in row
        }

    return {
        "id": pair.get("id", row.get("idx")),
        "idx": row.get("idx"),
        "source_group": source_group,
        "C1": pair.get("C1", row.get("C1", "")),
        "C2": pair.get("C2", row.get("C2", "")),
        "T1": pair.get("T1", row.get("T1", "")),
        "T2": pair.get("T2", row.get("T2", "")),
        "A1_material_class": pair.get("A1_material_class", row.get("A1_material_class", "")),
        "A2_material_class": pair.get("A2_material_class", row.get("A2_material_class", "")),
        "key_material_noun": pair.get("key_material_noun", row.get("key_material_noun", "")),
        "notes": pair.get("notes") or row.get("verifier_detail", ""),
        # Optional but helpful; safe to drop later
        "verifier": verifier,
    }


def _pair_from_repaired_row(row: dict, source_group: str = "repaired") -> dict:
    repaired = row.get("repaired", {}) or {}
    input_item = row.get("input_item", {}) or {}

    return {
        "id": repaired.get("id", input_item.get("id", row.get("idx"))),
        "idx": row.get("idx"),
        "source_group": source_group,
        "C1": repaired.get("C1", input_item.get("C1", "")),
        "C2": repaired.get("C2", input_item.get("C2", "")),
        "T1": repaired.get("T1", input_item.get("T1", "")),
        "T2": repaired.get("T2", input_item.get("T2", "")),
        "A1_material_class": repaired.get("A1_material_class", input_item.get("A1_material_class", "")),
        "A2_material_class": repaired.get("A2_material_class", input_item.get("A2_material_class", "")),
        "key_material_noun": repaired.get("key_material_noun", input_item.get("key_material_noun", "")),
        "notes": repaired.get("notes", input_item.get("notes", "")),
        # Keep the verifier snapshot that was fed to the editor (if you want it)
        "verifier": input_item.get("verifier", {}),
    }


def _dedupe_pairs(pairs: list[dict]) -> list[dict]:
    seen = set()
    out = []
    for p in pairs:
        key = (
            p.get("id"),
            p.get("C1"),
            p.get("C2"),
            p.get("T1"),
            p.get("T2"),
        )
        if key in seen:
            continue
        seen.add(key)
        out.append(p)
    return out


def _write_jsonl(path: Path, rows: list[dict]) -> None:
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


if "compact_partitioned_results_payload" not in globals():
    raise NameError("compact_partitioned_results_payload not found. Run the compacting cell first.")
if "editor_gpt52_payload" not in globals():
    raise NameError("editor_gpt52_payload not found. Run the editor repair cell first.")

passed_rows = compact_partitioned_results_payload.get("passed_verifier", [])
repaired_rows = editor_gpt52_payload.get("repaired", [])

passed_pairs_for_aug = [_pair_from_compact_row(r, "passed_verifier") for r in passed_rows]
repaired_pairs_for_aug = [_pair_from_repaired_row(r, "repaired") for r in repaired_rows]
merged_pairs_for_aug = _dedupe_pairs(passed_pairs_for_aug + repaired_pairs_for_aug)

out_dir = Path("data_augmentation")
out_dir.mkdir(parents=True, exist_ok=True)

passed_jsonl = out_dir / "context_target_pairs_passed.jsonl"
repaired_jsonl = out_dir / "context_target_pairs_repaired.jsonl"
merged_jsonl = out_dir / "context_target_pairs_merged.jsonl"
merged_json = out_dir / "context_target_pairs_merged.json"
manifest_json = out_dir / "manifest.json"

_write_jsonl(passed_jsonl, passed_pairs_for_aug)
_write_jsonl(repaired_jsonl, repaired_pairs_for_aug)
_write_jsonl(merged_jsonl, merged_pairs_for_aug)
merged_json.write_text(json.dumps(merged_pairs_for_aug, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "output_dir": str(out_dir),
    "counts": {
        "passed_pairs": len(passed_pairs_for_aug),
        "repaired_pairs": len(repaired_pairs_for_aug),
        "merged_pairs": len(merged_pairs_for_aug),
    },
    "files": {
        "passed_jsonl": str(passed_jsonl),
        "repaired_jsonl": str(repaired_jsonl),
        "merged_jsonl": str(merged_jsonl),
        "merged_json": str(merged_json),
    },
}
manifest_json.write_text(json.dumps(manifest, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

manifest

In [ ]:
import re
import json
from pathlib import Path

# Concepts to track (normalized)
CONCEPTS = [
    "wrinkle", "squeeze", "pour", "drape", "tap", "pile", "fold", "stir",
    "hang", "rip", "splash", "ripple", "drip", "break", "flap",
]


def _load_pairs_for_concept_stats() -> list[dict]:
    # 1) Use in-memory merged pairs if available
    if "merged_pairs_for_aug" in globals() and isinstance(merged_pairs_for_aug, list):
        return merged_pairs_for_aug

    # 2) Fallback to merged JSON file written by previous cell
    merged_json_path = Path("data_augmentation/context_target_pairs_merged.json")
    if merged_json_path.exists():
        return json.loads(merged_json_path.read_text(encoding="utf-8"))

    # 3) Fallback to merged JSONL file
    merged_jsonl_path = Path("data_augmentation/context_target_pairs_merged.jsonl")
    if merged_jsonl_path.exists():
        rows = []
        with merged_jsonl_path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                rows.append(json.loads(line))
        return rows

    raise FileNotFoundError(
        "Could not find merged pairs in memory or data_augmentation files. Run the merge/export cell first."
    )


def _pair_text(pair: dict) -> str:
    return " ".join([
        str(pair.get("C1", "")),
        str(pair.get("C2", "")),
        str(pair.get("T1", "")),
        str(pair.get("T2", "")),
    ])


pairs = _load_pairs_for_concept_stats()
N = len(pairs)

rows = []
for concept in CONCEPTS:
    # whole-word, case-insensitive match
    pat = re.compile(r"\b" + re.escape(concept) + r"\b", flags=re.IGNORECASE)

    pairs_with_concept = 0
    total_occurrences = 0

    for pair in pairs:
        txt = _pair_text(pair)
        hits = pat.findall(txt)
        if hits:
            pairs_with_concept += 1
            total_occurrences += len(hits)

    pct_of_pairs = (100.0 * pairs_with_concept / N) if N else 0.0
    rows.append({
        "concept": concept,
        "pairs_with_concept": pairs_with_concept,
        "pct_of_all_pairs": round(pct_of_pairs, 2),
        "total_occurrences": total_occurrences,
    })

# Sort highest coverage first
rows = sorted(rows, key=lambda r: (-r["pairs_with_concept"], r["concept"]))

print(f"Total pairs: {N}")
rows



In [ ]:
import re

# Concept coverage analysis for mining/filter stages (C1+T1 only)
# Use inflection-aware patterns so forms like "stirring" and "hung" are counted.
CONCEPT_PATTERNS_STAGE = {
    "wrinkle": re.compile(r"\b(wrinkle|wrinkles|wrinkled|wrinkling)\b", re.I),
    "squeeze": re.compile(r"\b(squeeze|squeezes|squeezed|squeezing)\b", re.I),
    "pour": re.compile(r"\b(pour|pours|poured|pouring)\b", re.I),
    "drape": re.compile(r"\b(drape|drapes|draped|draping)\b", re.I),
    "tap": re.compile(r"\b(tap|taps|tapped|tapping)\b", re.I),
    "pile": re.compile(r"\b(pile|piles|piled|piling)\b", re.I),
    "fold": re.compile(r"\b(fold|folds|folded|folding)\b", re.I),
    "stir": re.compile(r"\b(stir|stirs|stirred|stirring)\b", re.I),
    "hang": re.compile(r"\b(hang|hangs|hung|hanging)\b", re.I),
    "rip": re.compile(r"\b(rip|rips|ripped|ripping)\b", re.I),
    "splash": re.compile(r"\b(splash|splashes|splashed|splashing)\b", re.I),
    "ripple": re.compile(r"\b(ripple|ripples|rippled|rippling)\b", re.I),
    "drip": re.compile(r"\b(drip|drips|dripped|dripping)\b", re.I),
    "break": re.compile(r"\b(break|breaks|broke|broken|breaking)\b", re.I),
    "flap": re.compile(r"\b(flap|flaps|flapped|flapping)\b", re.I),
}


def _looks_like_c1_t1_items(items):
    if not isinstance(items, list):
        return False
    if len(items) == 0:
        return False
    first = items[0]
    return isinstance(first, dict) and ("C1" in first) and ("T1" in first)


def _resolve_pairs_stage_items():
    # Prefer in-memory `pairs` if it still refers to mined C1/T1 items.
    # (Later cells may overwrite `pairs` with merged C1/C2/T1/T2 items.)
    if "pairs" in globals() and _looks_like_c1_t1_items(pairs):
        first = pairs[0]
        if not ("C2" in first and "T2" in first):
            return pairs

    # Fallback: recompute from all_rows if available
    if "all_rows" in globals() and "mine_C1_T1_pairs" in globals():
        return mine_C1_T1_pairs(all_rows, require_cue=True)

    raise NameError(
        "Could not resolve stage-1 `pairs`. Run mining cells (all_rows + mine_C1_T1_pairs) first."
    )


def _resolve_kept2_stage_items(pairs_stage):
    # Prefer in-memory kept2 if present
    if "kept2" in globals() and _looks_like_c1_t1_items(kept2):
        return kept2

    # Fallback: recompute from pairs_stage
    if "filter_items" in globals() and "filter_for_contrast_pairs" in globals():
        kept_stage, _ = filter_items(
            pairs_stage,
            require_swappable_noun=True,
            require_noun_in_context=True,
            require_literal_frame=True,
            require_cue_for_literal=True,
            attach_metadata=True,
        )
        kept2_stage, _ = filter_for_contrast_pairs(
            kept_stage,
            allowed_verbs=["stir", "wrinkle", "hang", "squeeze"],
            disallow_multi_verb=True,
        )
        return kept2_stage

    raise NameError(
        "Could not resolve stage-2 `kept2`. Run filtering cells (filter_items + filter_for_contrast_pairs) first."
    )


def _concept_stats_c1_t1(items, concept_patterns):
    N = len(items)
    rows = []

    for concept, pat in concept_patterns.items():
        pairs_with_concept = 0
        total_occurrences = 0

        for item in items:
            text = f"{item.get('C1', '')} {item.get('T1', '')}"
            hits = pat.findall(text)
            if hits:
                pairs_with_concept += 1
                total_occurrences += len(hits)

        rows.append({
            "concept": concept,
            "pairs_with_concept": pairs_with_concept,
            "pct_of_all_pairs": round((100.0 * pairs_with_concept / N) if N else 0.0, 2),
            "total_occurrences": total_occurrences,
        })

    rows = sorted(rows, key=lambda r: (-r["pairs_with_concept"], r["concept"]))
    return rows


pairs_stage = _resolve_pairs_stage_items()
kept2_stage = _resolve_kept2_stage_items(pairs_stage)

pairs_stats = _concept_stats_c1_t1(pairs_stage, CONCEPT_PATTERNS_STAGE)
kept2_stats = _concept_stats_c1_t1(kept2_stage, CONCEPT_PATTERNS_STAGE)

kept2_map = {r["concept"]: r for r in kept2_stats}
filter_delta = []
for r in pairs_stats:
    k = kept2_map[r["concept"]]
    filtered_out_count = r["pairs_with_concept"] - k["pairs_with_concept"]
    filtered_out_pct_of_pairs_concept = (
        round(100.0 * filtered_out_count / r["pairs_with_concept"], 2)
        if r["pairs_with_concept"] else 0.0
    )
    retained_pct_of_pairs_concept = (
        round(100.0 * k["pairs_with_concept"] / r["pairs_with_concept"], 2)
        if r["pairs_with_concept"] else 0.0
    )

    filter_delta.append({
        "concept": r["concept"],
        "pairs_count": r["pairs_with_concept"],
        "pairs_pct": r["pct_of_all_pairs"],
        "kept2_count": k["pairs_with_concept"],
        "kept2_pct": k["pct_of_all_pairs"],
        "filtered_out_count": filtered_out_count,
        "filtered_out_pct_of_pairs_concept": filtered_out_pct_of_pairs_concept,
        "retained_pct_of_pairs_concept": retained_pct_of_pairs_concept,
    })

concept_filter_analysis = {
    "totals": {
        "pairs_total": len(pairs_stage),
        "kept2_total": len(kept2_stage),
    },
    "pairs_stats": pairs_stats,
    "kept2_stats": kept2_stats,
    "filter_delta": filter_delta,
}

print(f"Stage totals: pairs={len(pairs_stage)}, kept2={len(kept2_stage)}")
print("\nPairs stats (C1+T1):")
pairs_stats



In [ ]:
import re
from collections import Counter

# Better concept counting for the final merged export.
# Prefer explicit ACTIVE=<concept> tags when present, then fall back to
# inflection-aware matching over the full C1/C2/T1/T2 text.

pairs_final = _load_pairs_for_concept_stats()

ACTIVE_CONCEPT_PATTERNS_FINAL = {
    "wrinkle": re.compile(r"\b(wrinkle|wrinkles|wrinkled|wrinkling)\b", re.I),
    "squeeze": re.compile(r"\b(squeeze|squeezes|squeezed|squeezing)\b", re.I),
    "stir": re.compile(r"\b(stir|stirs|stirred|stirring)\b", re.I),
    "hang": re.compile(r"\b(hang|hangs|hung|hanging)\b", re.I),
}

NOTE_ACTIVE_RE = re.compile(r"\bACTIVE=(stir|squeeze|hang|wrinkle)\b", re.I)


def _pair_text_full(pair: dict) -> str:
    return " ".join([
        str(pair.get("C1", "")),
        str(pair.get("C2", "")),
        str(pair.get("T1", "")),
        str(pair.get("T2", "")),
    ])


def _match_counts(text: str, patterns: dict[str, re.Pattern]) -> dict[str, int]:
    return {concept: len(pat.findall(text)) for concept, pat in patterns.items()}


assignment_rows = []
assigned_counts = Counter()
assignment_source_counts = Counter()

for idx, pair in enumerate(pairs_final):
    notes = str(pair.get("notes", ""))
    text = _pair_text_full(pair)

    note_match = NOTE_ACTIVE_RE.search(notes)
    active_from_notes = note_match.group(1).lower() if note_match else None

    match_counts = _match_counts(text, ACTIVE_CONCEPT_PATTERNS_FINAL)
    matched_active = [concept for concept, count in match_counts.items() if count > 0]

    assigned_concept = None
    assignment_source = None

    if active_from_notes:
        assigned_concept = active_from_notes
        assignment_source = "notes_active"
    elif len(matched_active) == 1:
        assigned_concept = matched_active[0]
        assignment_source = "text_unique"
    elif len(matched_active) > 1:
        best_count = max(match_counts[c] for c in matched_active)
        top = [c for c in matched_active if match_counts[c] == best_count]
        if len(top) == 1:
            assigned_concept = top[0]
            assignment_source = "text_max_count"
        else:
            assignment_source = "ambiguous"
    else:
        assignment_source = "no_active_match"

    if assigned_concept:
        assigned_counts[assigned_concept] += 1

    assignment_source_counts[assignment_source] += 1

    assignment_rows.append({
        "idx": pair.get("idx", idx),
        "id": pair.get("id"),
        "source_group": pair.get("source_group"),
        "assigned_concept": assigned_concept,
        "assignment_source": assignment_source,
        "active_from_notes": active_from_notes,
        "text_match_counts": match_counts,
        "matched_active": matched_active,
        "C1": pair.get("C1", ""),
        "T1": pair.get("T1", ""),
        "notes": notes,
    })

better_concept_count_summary = {
    "total_pairs": len(pairs_final),
    "assigned_pairs": sum(assigned_counts.values()),
    "unassigned_pairs": len(pairs_final) - sum(assigned_counts.values()),
    "assigned_counts": dict(sorted(assigned_counts.items())),
    "assignment_source_counts": dict(sorted(assignment_source_counts.items())),
}

ambiguous_or_unassigned_pairs = [
    row for row in assignment_rows
    if row["assigned_concept"] is None
]

print("Better concept-count summary for merged pairs:")
print(better_concept_count_summary)
print("\nAssigned concept counts:")
for concept in sorted(ACTIVE_CONCEPT_PATTERNS_FINAL):
    print(f"  {concept}: {assigned_counts.get(concept, 0)}")

print("\nAmbiguous / unassigned sample (up to 10 rows):")
ambiguous_or_unassigned_pairs[:10]


In [ ]:
import matplotlib.pyplot as plt

if "assigned_counts" not in globals():
    raise NameError("assigned_counts not found. Run the better concept-count cell first.")

plot_rows = sorted(assigned_counts.items(), key=lambda kv: kv[1], reverse=True)
labels = [k for k, _ in plot_rows]
values = [v for _, v in plot_rows]

plt.figure(figsize=(8, 4.5))
bars = plt.bar(labels, values)
plt.title("Assigned concept counts in merged minimal pairs")
plt.xlabel("Concept")
plt.ylabel("Count")

for bar, value in zip(bars, values):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.3,
        str(value),
        ha="center",
        va="bottom",
    )

plt.ylim(0, max(values) + 4 if values else 1)
plt.tight_layout()
plt.show()


In [ ]:
if "pairs_stage" not in globals():
    pairs_stage = _resolve_pairs_stage_items()
if "kept2_stage" not in globals():
    kept2_stage = _resolve_kept2_stage_items(pairs_stage)
if "compact_partitioned_results_payload" not in globals():
    raise NameError("compact_partitioned_results_payload not found. Run the verifier compaction cell first.")
if "editor_gpt52_payload" not in globals():
    raise NameError("editor_gpt52_payload not found. Run the editor repair cell first.")

pairs_total = len(pairs_stage)
kept_total = len(kept2_stage)
filtered_out_before_generation = pairs_total - kept_total

partition_counts = compact_partitioned_results_payload["counts"]
passed_direct = partition_counts["passed_verifier"]
model_rejects = partition_counts["failed_model_reject"]
repair_candidates = partition_counts["failed_status_ok"]

repair_counts = editor_gpt52_payload["counts"]
repaired = repair_counts["repaired"]
repair_rejected = repair_counts["rejected"]
repair_parse_error = repair_counts["parse_error"]

final_usable = passed_direct + repaired

pipeline_summary = {
    "pairs_total": pairs_total,
    "kept_for_generation": kept_total,
    "filtered_out_before_generation": filtered_out_before_generation,
    "passed_direct": passed_direct,
    "model_rejects": model_rejects,
    "repair_candidates": repair_candidates,
    "repaired": repaired,
    "repair_rejected": repair_rejected,
    "repair_parse_error": repair_parse_error,
    "final_usable": final_usable,
}

print("Synthetic minimal-pair pipeline summary")
print(f"- Mined {pairs_total} possible (C1, T1) pairs from decoded exposure text.")
print(
    f"- Kept {kept_total} / {pairs_total} for GPT-5.2 generation "
    f"({100.0 * kept_total / pairs_total:.2f}%), after filtering out {filtered_out_before_generation} "
    f"literal/material mismatches ({100.0 * filtered_out_before_generation / pairs_total:.2f}%)."
)
print(
    f"- Among the {kept_total} generated items: {passed_direct} passed the verifier immediately, "
    f"{model_rejects} were model-level rejects, and {repair_candidates} failed verifier checks "
    f"but were sent to repair."
)
print(
    f"- The repair stage fixed {repaired} / {repair_candidates} candidates "
    f"({100.0 * repaired / repair_candidates:.2f}%); {repair_rejected} were still rejected"
    + (f", and {repair_parse_error} hit parse errors." if repair_parse_error else ".")
)
print(
    f"- Final usable set: {final_usable} pairs = {passed_direct} direct passes + {repaired} repairs, "
    f"which is {100.0 * final_usable / pairs_total:.2f}% of the original {pairs_total} mined pairs "
    f"and {100.0 * final_usable / kept_total:.2f}% of the kept-for-generation pool."
)

pipeline_summary


In [ ]:
import matplotlib.pyplot as plt

if "pipeline_summary" not in globals():
    raise NameError("pipeline_summary not found. Run the pipeline summary cell first.")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

# Left: stage sizes
stage_labels = ["Mined C1/T1", "Kept for generation", "Final usable"]
stage_values = [
    pipeline_summary["pairs_total"],
    pipeline_summary["kept_for_generation"],
    pipeline_summary["final_usable"],
]
bars = axes[0].bar(stage_labels, stage_values, color=["#7aa6c2", "#5d8aa8", "#3d5a80"])
axes[0].set_title("Pipeline stage sizes")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=15)
for bar, value in zip(bars, stage_values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, value + 10, str(value), ha="center", va="bottom")

# Right: breakdown of the kept-for-generation pool
breakdown_labels = ["Passed directly", "Model rejects", "Sent to repair"]
breakdown_values = [
    pipeline_summary["passed_direct"],
    pipeline_summary["model_rejects"],
    pipeline_summary["repair_candidates"],
]
bars2 = axes[1].bar(breakdown_labels, breakdown_values, color=["#2a9d8f", "#e76f51", "#e9c46a"])
axes[1].set_title("Breakdown of kept-for-generation items")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=15)
for bar, value in zip(bars2, breakdown_values):
    axes[1].text(bar.get_x() + bar.get_width() / 2, value + 2, str(value), ha="center", va="bottom")

fig.suptitle("From mined candidates to usable synthetic minimal pairs")
plt.tight_layout()
plt.show()
